# LeetCode 248 — Strobogrammatic Number III bằng Python

> **Lưu ý: đây là bài Premium** (bị khóa trên LeetCode, chỉ tài khoản trả phí mới mở được). Hai bài "anh em" đi trước nó — **246. Strobogrammatic Number** (Easy) và **247. Strobogrammatic Number II** (Medium) — cũng là bài Premium. Notebook này ôn lại cả hai bài đó **từ con số 0** trước khi giải bài 248, nên bạn không cần làm 246/247 trước.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def strobogrammaticInRange(self, low: str, high: str) -> int:

```

## Sau notebook này, bạn sẽ nắm được

1. **Strobogrammatic number** là gì: số trông y hệt khi **xoay ngược 180°** (lật tờ giấy lộn đầu). Bảng "xoay" của các chữ số và các cặp hợp lệ `0-0, 1-1, 6-9, 8-8, 9-6`, tâm chỉ được `0/1/8`.
2. Nền tảng Python và tư duy cần dùng: `dict` tra cứu, `reversed` + `join`, **hai con trỏ** (two pointers), **so sánh chuỗi** (và cái bẫy `"50" > "100"`), **đệ quy** (recursion), **quy tắc nhân** trong đếm tổ hợp, hàm lồng nhau.
3. Năm cách giải, đi từ trực giác tới khéo léo:
   - **Cách 1:** Brute force — duyệt **từng số** trong `[low, high]` và kiểm tra (LeetCode 246). Chỉ dùng được cho khoảng nhỏ, ta giữ nó làm **trọng tài** để kiểm tra các cách khác.
   - **Cách 2:** **Sinh** mọi số strobogrammatic theo từng độ dài bằng đệ quy **từ trong ra ngoài** (LeetCode 247), rồi lọc theo khoảng.
   - **Cách 3 (cách chính):** **DFS** đặt từng cặp chữ số **từ ngoài vào trong**, đếm trực tiếp, không cần lưu danh sách.
   - **Cách 4:** Độ dài nằm **hẳn giữa** `len(low)` và `len(high)` thì đếm bằng **công thức tổ hợp**; chỉ chạy DFS cho hai độ dài biên.
   - **Cách 5 (nâng cao):** Đếm **từng chữ số** — tính `F(x)` = số strobogrammatic trong `[0, x]` mà không sinh số nào, chỉ `O(L²)` phép tính.
4. Vì sao mỗi cách đúng, độ phức tạp, các lỗi hay gặp (so sánh chuỗi khác độ dài, số 0 đứng đầu, tâm dùng 6/9...).
5. Chuyển kỹ thuật sang các bài cùng họ: 246, 247, 1056, 1088, 902, 906.

## 1. Đề bài (theo LeetCode)

Given two strings `low` and `high` that represent two integers `low` and `high` where `low <= high`, return *the number of* ***strobogrammatic numbers*** *in the range* `[low, high]`.

A **strobogrammatic number** is a number that looks the same when rotated `180` degrees (looked at upside down).

**Example 1**

```text
Input:  low = "50", high = "100"
Output: 3
```

**Example 2**

```text
Input:  low = "0", high = "0"
Output: 1
```

**Constraints**

- `1 <= low.length, high.length <= 15`
- `low` and `high` consist of only digits.
- `low <= high`
- `low` and `high` do not contain any leading zeros except for zero itself.

### Dịch nhanh

- `low` và `high` là **chuỗi** (string) biểu diễn hai số nguyên, `low <= high` (so sánh theo **giá trị số**).
- Đếm xem trong đoạn `[low, high]` (lấy **cả hai đầu**) có bao nhiêu số **strobogrammatic** — số mà khi xoay ngược tờ giấy 180° vẫn đọc ra **chính nó**.
- Mỗi chuỗi dài tối đa **15** chữ số, tức `high` có thể lên tới `999 999 999 999 999` (gần `10^15`). Đây là gợi ý quan trọng: **không thể** duyệt từng số.
- "Không có số 0 đứng đầu, trừ chính số 0" (leading zeros): đầu vào kiểu `"0050"` không xuất hiện. Ta đếm **số nguyên**, nên `"00"`, `"0110"`... không phải là một số hợp lệ để đếm.

## 2. Hiểu đề bằng tay

### 2.1. Xoay 180° một chữ số

Hãy tưởng tượng viết chữ số lên giấy rồi xoay ngược tờ giấy (đầu xuống dưới):

| Chữ số | Xoay 180° thành | Còn là chữ số? |
|:---:|:---:|:---:|
| `0` | `0` | có |
| `1` | `1` | có (coi `1` là một nét thẳng) |
| `6` | `9` | có |
| `8` | `8` | có |
| `9` | `6` | có |
| `2`, `3`, `4`, `5`, `7` | hình lạ, không phải chữ số | **không** |

Chỉ **5** chữ số "sống sót" sau khi xoay: `0, 1, 6, 8, 9`. Số nào chứa `2, 3, 4, 5, 7` thì chắc chắn **không** strobogrammatic.

### 2.2. Xoay 180° cả một số = đọc ngược + lật từng chữ số

Khi xoay ngược tờ giấy, có **hai** việc xảy ra cùng lúc:

1. **Thứ tự đảo ngược**: chữ số ở cuối bên phải chạy lên đầu bên trái.
2. **Mỗi chữ số bị lật**: `6 ↔ 9`, còn `0, 1, 8` giữ nguyên.

| Số | Đọc ngược | Lật từng chữ | Kết quả xoay | Giống số gốc? |
|---|---|---|---|:---:|
| `69` | `96` | `9→6, 6→9` | `69` | **có** |
| `88` | `88` | `8→8, 8→8` | `88` | **có** |
| `619` | `916` | `9→6, 1→1, 6→9` | `619` | **có** |
| `1968` | `8691` | `8→8, 6→9, 9→6, 1→1` | `8961` | không |
| `100` | `001` | `0, 0, 1` | `001` | không |
| `6` | `6` | `6→9` | `9` | không |

Chú ý dòng cuối: **`6` đứng một mình không strobogrammatic** (xoay thành `9`). Số có **1 chữ số** chỉ có `0`, `1`, `8`.

### 2.3. Giải Example 1 bằng tay: `low = "50"`, `high = "100"`

- Số 2 chữ số strobogrammatic: chữ số đầu và chữ số cuối phải là một **cặp xoay** `(a, xoay(a))`, và chữ số đầu không được là `0`: `11, 69, 88, 96`. Những số `>= 50`: **`69, 88, 96`**.
- Số 3 chữ số `<= 100`: chỉ có chính `100`, mà `100` xoay thành `001` ⇒ không tính.
- Đáp án: **3**. Example 2: chỉ có số `0`, và `0` xoay vẫn là `0` ⇒ **1**.

Trước khi học kỹ thuật, nhìn một "cái bẫy" có ngay trong đề: `low`, `high` là **chuỗi**, không phải số.

In [1]:
low, high = "50", "100"
print(type(low), type(high))
print("So sánh như SỐ    :", int(low) <= int(high))   # 50 <= 100 -> True
print("So sánh như CHUỖI :", low <= high)             # "50" <= "100" -> False (!)

<class 'str'> <class 'str'>
So sánh như SỐ    : True
So sánh như CHUỖI : False


Cùng một cặp giá trị mà so sánh kiểu số ra `True`, so sánh kiểu chuỗi lại ra `False`. Lý do: chuỗi được so **từng ký tự từ trái sang**, và `'5' > '1'` nên `"50" > "100"`. Mục 3.4 sẽ giải thích kỹ, vì đây là lỗi số 1 của bài này.

Vài **edge case** (trường hợp biên) cần ghi nhớ ngay từ đầu:

- `low = high` (đoạn chỉ có 1 số): đáp án là `1` nếu số đó strobogrammatic, ngược lại `0`.
- `low = "0"`: số `0` **được** tính (Example 2).
- Số có tâm (độ dài lẻ) thì chữ số ở tâm xoay xong phải là **chính nó** ⇒ chỉ `0, 1, 8`. Ví dụ `161` **không** hợp lệ (tâm `6` xoay thành `9`).

## 3. Nền tảng (học từ con số 0)

### 3.1. `dict` — bảng tra "chữ số này xoay thành gì?"

**`dict`** (dictionary, "từ điển") lưu các cặp **key → value**. Ở bài này: key là chữ số gốc, value là chữ số sau khi xoay 180°.

- Tạo: `ROT = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}`.
- Tra cứu: `ROT["6"]` cho `"9"`.
- Kiểm tra có key không: `"2" in ROT` cho `False` (chữ số `2` không xoay được).
- Tra một key **không có** bằng `ROT["2"]` sẽ báo lỗi `KeyError`; muốn an toàn thì dùng `ROT.get("2")`, trả về `None` thay vì báo lỗi.

Ta lưu chữ số dưới dạng **chuỗi 1 ký tự** (`"6"`, không phải `6`) vì `low`, `high` và các số ta sinh ra đều là chuỗi.

In [2]:
ROT = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}

print(ROT["6"], ROT["9"], ROT["8"])    # 9 6 8
print("2" in ROT, "0" in ROT)          # False True
print(ROT.get("2"))                    # None (không báo lỗi)
print(list(ROT.keys()))                # 5 chữ số "sống sót" sau khi xoay

9 6 8
False True
None
['0', '1', '6', '8', '9']


### 3.2. Xoay cả một số: `reversed` + lật từng chữ + `"".join`

Theo mục 2.2, xoay = **đọc ngược** + **lật từng chữ số**. Python có sẵn:

- `reversed(s)`: duyệt chuỗi `s` từ **cuối về đầu**. Ví dụ `reversed("1968")` lần lượt cho `'8', '6', '9', '1'`.
- `"".join(list_ky_tu)`: dán các ký tự trong list thành một chuỗi. Ví dụ `"".join(["8", "9", "6", "1"])` cho `"8961"`.

Hàm `rotate(s)` dưới đây trả về chuỗi sau khi xoay, hoặc `None` nếu gặp chữ số không xoay được. Số `s` là strobogrammatic **khi và chỉ khi** `rotate(s) == s`.

In [3]:
def rotate(s: str):
    out = []
    for ch in reversed(s):          # đọc ngược
        if ch not in ROT:
            return None             # có chữ số 2/3/4/5/7 -> không xoay được
        out.append(ROT[ch])         # lật từng chữ số
    return "".join(out)


for s in ["69", "88", "619", "1968", "100", "6", "25"]:
    r = rotate(s)
    print(f"{s:>5} -> xoay thành {r!s:>5} | strobogrammatic: {r == s}")

   69 -> xoay thành    69 | strobogrammatic: True
   88 -> xoay thành    88 | strobogrammatic: True
  619 -> xoay thành   619 | strobogrammatic: True
 1968 -> xoay thành  8961 | strobogrammatic: False
  100 -> xoay thành   001 | strobogrammatic: False
    6 -> xoay thành     9 | strobogrammatic: False
   25 -> xoay thành  None | strobogrammatic: False


### 3.3. Kiểm tra một số (LeetCode 246) bằng **hai con trỏ**

`rotate(s) == s` đúng nhưng phải tạo thêm một chuỗi mới. Cách gọn hơn và quan trọng hơn cho phần sau là **hai con trỏ** (two pointers): một con trỏ `i` đi từ **đầu**, một con trỏ `j` đi từ **cuối**, cùng tiến vào giữa.

Vì xoay đưa chữ số ở vị trí `i` sang vị trí `j = n - 1 - i` (và lật nó), điều kiện strobogrammatic chính là:

> Với mọi cặp vị trí đối xứng `(i, j)`: `ROT[num[i]] == num[j]`.

Khi `i == j` (ô **tâm** của số có độ dài lẻ), điều kiện trở thành `ROT[c] == c` ⇒ tự động chỉ cho phép `0, 1, 8`. Không cần viết luật riêng cho tâm!

Bảng tay với `num = "16891"` và `num = "161"`:

| num | Bước | i | j | num[i] | num[j] | ROT[num[i]] | Khớp? |
|---|---|:-:|:-:|:-:|:-:|:-:|:-:|
| `16891` | 1 | 0 | 4 | 1 | 1 | 1 | có |
| `16891` | 2 | 1 | 3 | 6 | 9 | 9 | có |
| `16891` | 3 | 2 | 2 | 8 | 8 | 8 | có (tâm) ⇒ **True** |
| `161` | 1 | 0 | 2 | 1 | 1 | 1 | có |
| `161` | 2 | 1 | 1 | 6 | 6 | 9 | **không** (tâm 6) ⇒ **False** |

LeetCode 246 có signature `def isStrobogrammatic(self, num: str) -> bool`. Ta viết bản hàm `is_strobogrammatic` và một bản `trace_` in từng bước.

In [4]:
def is_strobogrammatic(num: str) -> bool:
    i, j = 0, len(num) - 1
    while i <= j:                                   # i == j vẫn phải kiểm (ô tâm)
        if num[i] not in ROT or ROT[num[i]] != num[j]:
            return False
        i += 1
        j -= 1
    return True


def trace_is_strobogrammatic(num: str) -> bool:
    i, j = 0, len(num) - 1
    while i <= j:
        a, b = num[i], num[j]
        want = ROT.get(a)
        ok = want == b
        print(f"  i={i}, j={j}: num[i]={a}, num[j]={b}, ROT[num[i]]={want} -> {'khớp' if ok else 'KHÔNG khớp'}")
        if not ok:
            return False
        i, j = i + 1, j - 1
    return True


for num in ["16891", "161", "962"]:
    print(f"num = {num!r}")
    print("  => kết quả:", trace_is_strobogrammatic(num))

num = '16891'
  i=0, j=4: num[i]=1, num[j]=1, ROT[num[i]]=1 -> khớp
  i=1, j=3: num[i]=6, num[j]=9, ROT[num[i]]=9 -> khớp
  i=2, j=2: num[i]=8, num[j]=8, ROT[num[i]]=8 -> khớp
  => kết quả: True
num = '161'
  i=0, j=2: num[i]=1, num[j]=1, ROT[num[i]]=1 -> khớp
  i=1, j=1: num[i]=6, num[j]=6, ROT[num[i]]=9 -> KHÔNG khớp
  => kết quả: False
num = '962'
  i=0, j=2: num[i]=9, num[j]=2, ROT[num[i]]=6 -> KHÔNG khớp
  => kết quả: False


### 3.4. So sánh chuỗi (lexicographic) — và cái bẫy `"50" > "100"`

Python so sánh hai chuỗi theo **thứ tự từ điển** (lexicographic):

1. So ký tự đầu tiên của hai chuỗi; ký tự nào "nhỏ hơn" thì chuỗi đó nhỏ hơn. Với chữ số: `'0' < '1' < ... < '9'`.
2. Nếu bằng nhau thì so tiếp ký tự thứ hai, thứ ba...
3. Nếu một chuỗi là **phần đầu** của chuỗi kia (ví dụ `"100"` và `"1000"`) thì chuỗi **ngắn hơn** nhỏ hơn.

Hệ quả:

- **Cùng độ dài** (và không có số 0 đứng đầu): thứ tự chuỗi **trùng** thứ tự số. Vì chữ số khác nhau đầu tiên (tính từ trái) nằm ở hàng cao nhất, nó quyết định tất cả: `6xxx < 8xxx` dù `xxx` là gì (`6999 < 8000`).
- **Khác độ dài**: thứ tự chuỗi có thể **sai** so với số: `"9" > "10"`, `"50" > "100"`.

Cách so sánh an toàn cho chuỗi số không có số 0 đứng đầu: so **độ dài trước**, rồi mới so chuỗi. Viết gọn bằng **tuple** `(len(s), s)`: Python so tuple từng phần tử từ trái sang, y như chuỗi.

Trong lời giải, ta sẽ chỉ so sánh chuỗi khi **cùng độ dài** — nên dùng thẳng `s < low` là an toàn.

In [5]:
print('"69" < "88"   :', "69" < "88")        # True  (cùng độ dài: đúng như số)
print('"50" < "100"  :', "50" < "100")       # False (khác độ dài: SAI so với số!)
print('"9"  < "10"   :', "9" < "10")         # False (sai so với số)
print('"100" < "1000":', "100" < "1000")     # True  (chuỗi ngắn là phần đầu)


def num_key(s: str):
    return (len(s), s)                       # so độ dài trước, rồi so chuỗi


print("num_key('50') < num_key('100'):", num_key("50") < num_key("100"))   # True: đúng như số
print(sorted(["100", "9", "50", "88", "1000"], key=num_key))

"69" < "88"   : True
"50" < "100"  : False
"9"  < "10"   : False
"100" < "1000": True
num_key('50') < num_key('100'): True
['9', '50', '88', '100', '1000']


### 3.5. Đệ quy (recursion) — hàm tự gọi chính nó

**Đệ quy** là khi một hàm giải bài toán lớn bằng cách gọi **chính nó** cho một bài toán **nhỏ hơn**. Một hàm đệ quy luôn có 2 phần:

1. **Trường hợp cơ sở** (base case): bài toán đủ nhỏ để trả lời ngay, **không** gọi tiếp. Thiếu phần này thì hàm gọi mãi không dừng.
2. **Bước đệ quy** (recursive case): giả sử lời gọi nhỏ hơn **đã trả đúng**, dùng kết quả đó để xây đáp án lớn.

Ví dụ mini: sinh mọi chuỗi độ dài `k` chỉ gồm chữ `a`, `b`.

- Cơ sở: `k = 0` ⇒ chỉ có chuỗi rỗng `[""]`.
- Đệ quy: mỗi chuỗi dài `k` = **một ký tự** (`a` hoặc `b`) + **một chuỗi dài `k - 1`** (lấy từ lời gọi nhỏ hơn).

Tham số `depth` chỉ để thụt lề khi in, cho thấy các lời gọi **chồng lên nhau** (call stack — "ngăn xếp lời gọi"): lời gọi `k = 2` phải chờ `k = 1` xong, `k = 1` phải chờ `k = 0` xong.

In [6]:
def all_strings(k: int, depth: int = 0) -> list:
    pad = "    " * depth
    print(f"{pad}gọi all_strings({k})")
    if k == 0:                                  # trường hợp cơ sở
        print(f"{pad}-> ['']")
        return [""]
    result = []
    for rest in all_strings(k - 1, depth + 1):  # tin rằng lời gọi nhỏ trả đúng
        for c in "ab":
            result.append(c + rest)
    print(f"{pad}-> {result}")
    return result


_ = all_strings(2)                          # gán vào _ để Jupyter không in lại list trả về

gọi all_strings(2)
    gọi all_strings(1)
        gọi all_strings(0)
        -> ['']
    -> ['a', 'b']
-> ['aa', 'ba', 'ab', 'bb']


### 3.6. Đệ quy "từ trong ra ngoài": bọc một cặp quanh lõi

Với các số **đối xứng**, có một kiểu đệ quy rất đẹp: thay vì thêm 1 ký tự ở đầu, ta **bọc một cặp** ký tự quanh một **lõi** (core) ngắn hơn 2 ký tự.

Ví dụ mini: sinh mọi **palindrome** (chuỗi đọc xuôi ngược như nhau) độ dài `k` từ chữ `a`, `b`:

- Palindrome dài `k` = `c` + (palindrome dài `k - 2`) + `c`, với `c` là `a` hoặc `b`.
- Vì mỗi lần giảm **2**, ta cần **hai** trường hợp cơ sở: `k = 0` ⇒ `[""]`, `k = 1` ⇒ `["a", "b"]` (chữ ở tâm).

```text
k = 3:  lõi dài 1 ∈ {a, b}   ->  a·a·a, b·a·b, a·b·a, b·b·b
k = 4:  lõi dài 2 ∈ {aa, bb} ->  a·aa·a, b·aa·b, a·bb·a, b·bb·b
```

Strobogrammatic cũng có đúng cấu trúc này, chỉ khác: cặp bọc là `(a, ROT[a])` thay vì `(c, c)`, và tâm là `0/1/8`. Đó chính là ý tưởng của LeetCode 247 mà ta dùng ở Cách 2.

In [7]:
def build_palindromes(k: int) -> list:
    if k == 0:
        return [""]
    if k == 1:
        return ["a", "b"]
    result = []
    for mid in build_palindromes(k - 2):   # lõi ngắn hơn 2 ký tự
        for c in "ab":
            result.append(c + mid + c)     # bọc một cặp quanh lõi
    return result


for k in range(5):
    print(k, build_palindromes(k))

0 ['']
1 ['a', 'b']
2 ['aa', 'bb']
3 ['aaa', 'bab', 'aba', 'bbb']
4 ['aaaa', 'baab', 'abba', 'bbbb']


### 3.7. Quy tắc nhân và quy tắc cộng — đếm mà không liệt kê

Đây là hai quy tắc đếm cơ bản ta sẽ dùng ở Cách 4 và Cách 5.

- **Quy tắc nhân** (product rule): nếu một lựa chọn gồm nhiều **bước độc lập**, bước 1 có `m` cách, bước 2 có `p` cách, ... thì tổng số cách là `m × p × ...`.
  - Ví dụ: mã khóa 3 ô, mỗi ô chọn một chữ trong `{1, 3, 5}` ⇒ `3 × 3 × 3 = 27` mã.
  - Nếu ô đầu **cấm** chữ `1` ⇒ `2 × 3 × 3 = 18` mã.
- **Quy tắc cộng** (sum rule): nếu các trường hợp **không chồng lên nhau** (rời nhau), tổng = cộng từng trường hợp. Ví dụ: số strobogrammatic dài 1, dài 2, dài 3... là các nhóm rời nhau ⇒ cộng số lượng từng độ dài.

`itertools.product("135", repeat=3)` (thư viện chuẩn) liệt kê **mọi** bộ 3 ô — dùng để kiểm chứng công thức bằng cách đếm thật.

In [8]:
from itertools import product

all_codes = ["".join(t) for t in product("135", repeat=3)]
print(len(all_codes), "=", 3 * 3 * 3, "| vài mã đầu:", all_codes[:5])

no_one_first = [c for c in all_codes if c[0] != "1"]
print(len(no_one_first), "=", 2 * 3 * 3)

27 = 27 | vài mã đầu: ['111', '113', '115', '131', '133']
18 = 18


### 3.8. Hàm lồng nhau và mảng "điền từ hai đầu vào giữa"

Ở Cách 3 ta sẽ điền một số dài `n` **theo cặp vị trí** `(left, right)`: `(0, n-1)`, rồi `(1, n-2)`, ... cho tới khi `left > right`. Với `n` lẻ, cặp cuối cùng có `left == right` — đó là **ô tâm**.

Hai công cụ Python cho việc này:

- **Mảng ký tự** `buf = [""] * n`: một list `n` ô, gán `buf[left], buf[right] = a, b` để đặt hai chữ số cùng lúc; cuối cùng `"".join(buf)` thành chuỗi.
- **Hàm lồng nhau** (nested function): định nghĩa hàm `go` **bên trong** hàm khác. Hàm trong **nhìn thấy** biến của hàm ngoài (`order`, `buf`, `n`...), nên không cần truyền qua tham số.
- Hàm trong **sửa nội dung** list của hàm ngoài (`order.append(...)`, `buf[0] = ...`) thoải mái. Nhưng nếu muốn **gán lại** một biến của hàm ngoài (ví dụ `count += 1` với `count` là số), phải khai báo `nonlocal count` ở đầu hàm trong; nếu quên, Python coi `count` là biến mới của hàm trong và báo lỗi `UnboundLocalError`.

In [9]:
def fill_order(n: int) -> list:
    order = []

    def go(left: int, right: int):          # hàm lồng: thấy được biến `order`
        if left > right:                    # đã điền hết -> dừng
            return
        order.append((left, right))
        go(left + 1, right - 1)             # tiến vào trong

    go(0, n - 1)
    return order


print("n = 4:", fill_order(4))
print("n = 5:", fill_order(5), "<- cặp cuối (2, 2) là ô tâm")

buf = [""] * 5
buf[0], buf[4] = "6", "9"
buf[1], buf[3] = "1", "1"
buf[2], buf[2] = "8", "8"                   # ô tâm: hai đầu trùng nhau
print(buf, "->", "".join(buf))


def count_calls(n: int) -> int:
    count = 0

    def go(left: int, right: int):
        nonlocal count                      # gán lại biến số của hàm ngoài -> cần nonlocal
        if left > right:
            return
        count += 1
        go(left + 1, right - 1)

    go(0, n - 1)
    return count


print("Số cặp vị trí khi n = 7:", count_calls(7))   # (0,6) (1,5) (2,4) (3,3) -> 4

n = 4: [(0, 3), (1, 2)]
n = 5: [(0, 4), (1, 3), (2, 2)] <- cặp cuối (2, 2) là ô tâm
['6', '1', '8', '1', '9'] -> 61819
Số cặp vị trí khi n = 7: 4


## 4. Cách 1 — Brute force: duyệt từng số rồi kiểm tra

Cách người mới nghĩ ra đầu tiên: "đề bảo đếm trong `[low, high]` thì cứ đi qua **từng số**, số nào strobogrammatic thì đếm".

1. Đổi `low`, `high` sang số nguyên bằng `int(...)`.
2. Duyệt `x` từ `int(low)` tới `int(high)` — nhớ `range(a, b)` **không lấy** `b`, nên phải viết `range(int(low), int(high) + 1)`.
3. Đổi `x` về chuỗi `str(x)` và kiểm tra bằng `is_strobogrammatic` (mục 3.3).

Ưu điểm: gần như không thể viết sai ⇒ rất hợp làm **trọng tài** (brute force để đối chiếu) cho các cách nhanh hơn trên khoảng nhỏ.

In [10]:
def strobo_range_bruteforce(low: str, high: str) -> int:
    count = 0
    for x in range(int(low), int(high) + 1):    # + 1 để lấy cả high
        if is_strobogrammatic(str(x)):
            count += 1
    return count


print(strobo_range_bruteforce("50", "100"))     # 3  (Example 1)
print(strobo_range_bruteforce("0", "0"))        # 1  (Example 2)
print(strobo_range_bruteforce("0", "100"))      # 7  (0, 1, 8, 11, 69, 88, 96)
print(strobo_range_bruteforce("10", "1000"))    # 16

3
1
7
16


### Cách 1 vướng ở đâu?

Đề cho `high` dài tới 15 chữ số, tức khoảng có thể chứa gần **`10^15`** số. Ô dưới đo tốc độ thật trên máy đang chạy rồi ước tính (con số thay đổi theo máy). Kể cả khi máy kiểm được vài triệu số mỗi giây, `10^15` số vẫn cần **nhiều năm** — trong khi LeetCode chỉ cho cỡ vài giây.

Quan sát thứ hai quan trọng hơn: trong `[0, 10^6)` chỉ có **199** số strobogrammatic. Ta kiểm tra 1 triệu số để tìm 199 số — gần **99,98%** công sức bị lãng phí. Số strobogrammatic **cực hiếm**, nên thay vì "duyệt rồi lọc", ta nên **sinh thẳng** những số đó.

In [11]:
import time

t0 = time.perf_counter()
found = strobo_range_bruteforce("0", "199999")
dt = time.perf_counter() - t0
rate = 200_000 / dt
years = 10**15 / rate / (3600 * 24 * 365)
print(f"Duyệt 200 000 số: {dt:.2f} giây (≈ {rate:,.0f} số/giây), tìm thấy {found} số strobogrammatic")
print(f"Ước tính duyệt 10^15 số: ≈ {years:,.0f} năm  -> không thể!")

useful = strobo_range_bruteforce("0", "999999")
print(f"Trong [0, 10^6): {useful} số có ích / 1 000 000 số đã kiểm = {useful / 10**6:.4%}")

Duyệt 200 000 số: 0.02 giây (≈ 8,187,488 số/giây), tìm thấy 124 số strobogrammatic
Ước tính duyệt 10^15 số: ≈ 4 năm  -> không thể!
Trong [0, 10^6): 199 số có ích / 1 000 000 số đã kiểm = 0.0199%


## 5. Bước đột phá: đừng lọc — hãy **sinh** thẳng các số strobogrammatic

### 5.1. Bóc từng lớp vỏ

Lấy một số strobogrammatic, ví dụ `16891`. Chữ số đầu `1` và chữ số cuối `1` là một **cặp xoay**. Bỏ cặp ngoài cùng đi (slicing `s[1:-1]` — bỏ ký tự đầu và ký tự cuối), phần còn lại `689` **vẫn** strobogrammatic. Bóc tiếp: `8` — một chữ số tâm.

```text
1 6 8 9 1   ->  cặp ngoài (1,1), lõi "689"
  6 8 9     ->  cặp ngoài (6,9), lõi "8"
    8       ->  tâm
```

Đảo chiều suy nghĩ: **mọi** số strobogrammatic dài `n` = **một cặp** `(a, ROT[a])` bọc quanh **một lõi** strobogrammatic dài `n - 2`. Chỉ có 5 cặp: `0-0, 1-1, 6-9, 8-8, 9-6`. Lõi cuối cùng là chuỗi rỗng (`n` chẵn) hoặc một chữ số tâm `0/1/8` (`n` lẻ).

Lưu ý: **lõi được phép bắt đầu bằng `0`**. Ví dụ `1001` có lõi `00`, `609` có lõi `0`. Nếu cấm `0` ở mọi lớp, ta sẽ bỏ sót những số này.

### 5.2. Hai luật đặc biệt: **số 0 đứng đầu** và **ô tâm**

1. **Lớp ngoài cùng không được là cặp `0-0`** (trừ khi số chỉ có 1 chữ số). Chuỗi `"0110"` qua được phép kiểm tra xoay, nhưng nó không phải cách viết hợp lệ của một số nguyên: `int("0110")` là `110`, và `110` **không** strobogrammatic. Riêng `"0"` (dài 1) thì hợp lệ — đó là số không (Example 2).
2. **Ô tâm** (độ dài lẻ) chỉ được `0, 1, 8` — những chữ số xoay ra chính nó. Đặt `6` ở tâm thì xoay xong thành `9`.

Ô dưới kiểm chứng cả hai luật.

In [12]:
def peel(s: str):
    while len(s) >= 2:
        print(f"  {s:>6} -> cặp ngoài ({s[0]}, {s[-1]}), ROT[{s[0]}] = {ROT[s[0]]}, lõi = {s[1:-1]!r}")
        s = s[1:-1]                          # bỏ ký tự đầu và cuối
    print(f"  {s!r:>6} -> lõi cuối cùng")


for s in ["16891", "1001", "609"]:
    print(f"Bóc {s}:")
    peel(s)


print('"0110" qua phép kiểm xoay?', is_strobogrammatic("0110"))
print('nhưng int("0110") =', int("0110"), "-> strobogrammatic?", is_strobogrammatic(str(int("0110"))))
print('"0" là số hợp lệ và strobogrammatic:', is_strobogrammatic("0"))

centers = [c for c in "0123456789" if ROT.get(c) == c]
print("Chữ số được đặt ở tâm:", centers)

Bóc 16891:
   16891 -> cặp ngoài (1, 1), ROT[1] = 1, lõi = '689'
     689 -> cặp ngoài (6, 9), ROT[6] = 9, lõi = '8'
     '8' -> lõi cuối cùng
Bóc 1001:
    1001 -> cặp ngoài (1, 1), ROT[1] = 1, lõi = '00'
      00 -> cặp ngoài (0, 0), ROT[0] = 0, lõi = ''
      '' -> lõi cuối cùng
Bóc 609:
     609 -> cặp ngoài (6, 9), ROT[6] = 9, lõi = '0'
     '0' -> lõi cuối cùng
"0110" qua phép kiểm xoay? True
nhưng int("0110") = 110 -> strobogrammatic? False
"0" là số hợp lệ và strobogrammatic: True
Chữ số được đặt ở tâm: ['0', '1', '8']


## 6. Cách 2 — Sinh mọi số theo từng độ dài (LeetCode 247), rồi lọc

### 6.1. Ôn LeetCode 247: sinh mọi số strobogrammatic dài `n`

Đề 247 (Premium, Medium): *Given an integer `n`, return all the strobogrammatic numbers that are of length `n`. You may return the answer in any order.* Ví dụ: `n = 2` ⇒ `["11","69","88","96"]`; `n = 1` ⇒ `["0","1","8"]`. Ràng buộc: `1 <= n <= 14`.

Ta viết hàm đệ quy `build_core(k, n)`, giống hệt `build_palindromes` ở mục 3.6:

- `k` = độ dài của **lõi** đang xây; `n` = độ dài **cuối cùng** của số (để biết khi nào đang ở lớp ngoài cùng).
- Cơ sở: `k == 0` ⇒ `[""]`; `k == 1` ⇒ `["0", "1", "8"]` (chữ số tâm).
- Đệ quy: lấy mọi lõi dài `k - 2`, bọc mỗi cặp trong `PAIRS` quanh lõi.
- Luật số 0 đứng đầu: **chỉ khi** `k == n` (lớp ngoài cùng) mới bỏ cặp `0-0`. Lớp trong vẫn giữ `0-0` (để có `1001`).

Khi `n == 1`, lời gọi `build_core(1, 1)` rơi vào cơ sở và trả về cả `"0"` — đúng, vì số `0` hợp lệ.

In [13]:
PAIRS = [("0", "0"), ("1", "1"), ("6", "9"), ("8", "8"), ("9", "6")]
CENTERS = ["0", "1", "8"]


def build_core(k: int, n: int) -> list:
    """Mọi chuỗi strobogrammatic dài k; n là độ dài cuối cùng của số."""
    if k == 0:
        return [""]
    if k == 1:
        return CENTERS[:]                      # bản sao, tránh sửa nhầm list gốc
    result = []
    for mid in build_core(k - 2, n):
        for a, b in PAIRS:
            if a == "0" and k == n:            # lớp ngoài cùng: cấm số 0 đứng đầu
                continue
            result.append(a + mid + b)
    return result


def find_strobogrammatic(n: int) -> list:      # LeetCode 247
    return build_core(n, n)


print(find_strobogrammatic(1))                 # ['0', '1', '8']
print(find_strobogrammatic(2))                 # ['11', '69', '88', '96']
print(find_strobogrammatic(3))
assert all(is_strobogrammatic(s) for n in range(1, 8) for s in find_strobogrammatic(n))

['0', '1', '8']
['11', '69', '88', '96']
['101', '609', '808', '906', '111', '619', '818', '916', '181', '689', '888', '986']


### 6.2. Trace `build_core(4, 4)`

| Lời gọi | Lõi nhận được | Cặp được bọc | Kết quả |
|---|---|---|---|
| `build_core(0)` | — (cơ sở) | — | `[""]` |
| `build_core(2, n=4)` — lớp trong | `[""]` | **5** cặp (giữ `0-0`) | `00, 11, 69, 88, 96` |
| `build_core(4, n=4)` — lớp ngoài cùng | 5 lõi | **4** cặp (bỏ `0-0`) | `5 × 4 = 20` số |

Hàm `trace_build_core` in đúng chuỗi lời gọi đó (thụt lề = độ sâu của call stack).

In [14]:
def trace_build_core(k: int, n: int, depth: int = 0) -> list:
    pad = "    " * depth
    if k == 0:
        print(f"{pad}build_core(0) = ['']  (cơ sở)")
        return [""]
    if k == 1:
        print(f"{pad}build_core(1) = ['0', '1', '8']  (cơ sở: ô tâm)")
        return CENTERS[:]
    print(f"{pad}build_core({k}, n={n}): cần các lõi dài {k - 2}")
    inner = trace_build_core(k - 2, n, depth + 1)
    result = [a + mid + b for mid in inner for a, b in PAIRS if not (a == "0" and k == n)]
    note = "lớp NGOÀI CÙNG: bỏ cặp 0-0" if k == n else "lớp trong: giữ cặp 0-0"
    print(f"{pad}build_core({k}) = {len(inner)} lõi × {len(result) // len(inner)} cặp = {len(result)} chuỗi ({note})")
    if len(result) <= 5:
        print(f"{pad}  {result}")
    return result


res = trace_build_core(4, 4)
res = sorted(res)
for i in range(0, len(res), 10):
    print("   ", " ".join(res[i:i + 10]))

build_core(4, n=4): cần các lõi dài 2
    build_core(2, n=4): cần các lõi dài 0
        build_core(0) = ['']  (cơ sở)
    build_core(2) = 1 lõi × 5 cặp = 5 chuỗi (lớp trong: giữ cặp 0-0)
      ['00', '11', '69', '88', '96']
build_core(4) = 5 lõi × 4 cặp = 20 chuỗi (lớp NGOÀI CÙNG: bỏ cặp 0-0)
    1001 1111 1691 1881 1961 6009 6119 6699 6889 6969
    8008 8118 8698 8888 8968 9006 9116 9696 9886 9966


### 6.3. Lọc theo khoảng `[low, high]`

Số strobogrammatic trong `[low, high]` chỉ có thể có độ dài từ `len(low)` tới `len(high)` (số ngắn hơn `low` thì nhỏ hơn `low`, số dài hơn `high` thì lớn hơn `high`, vì không có số 0 đứng đầu). Với mỗi độ dài `L`:

- Nếu `L == len(low)`: phải kiểm `s >= low` — **cùng độ dài** nên so chuỗi là an toàn (mục 3.4).
- Nếu `L == len(high)`: phải kiểm `s <= high` — cũng cùng độ dài.
- Nếu `len(low) < L < len(high)`: mọi số dài `L` đều nằm trong khoảng, **không cần so sánh gì**.
- Nếu `len(low) == len(high)`: chỉ có một độ dài, phải kiểm **cả hai** điều kiện.

Viết như sau là gói gọn cả bốn trường hợp:

```python
if L == len(low) and s < low:   continue   # nhỏ hơn low -> bỏ
if L == len(high) and s > high: continue   # lớn hơn high -> bỏ
count += 1
```

In [15]:
def strobo_range_generate(low: str, high: str) -> int:
    count = 0
    for L in range(len(low), len(high) + 1):
        for s in find_strobogrammatic(L):
            if L == len(low) and s < low:       # cùng độ dài với low -> so chuỗi an toàn
                continue
            if L == len(high) and s > high:     # cùng độ dài với high
                continue
            count += 1
    return count


for low, high in [("50", "100"), ("0", "0"), ("0", "100"), ("10", "1000"), ("609", "906")]:
    got = strobo_range_generate(low, high)
    assert got == strobo_range_bruteforce(low, high)
    print(f"[{low}, {high}] -> {got}")

[50, 100] -> 3
[0, 0] -> 1
[0, 100] -> 7
[10, 1000] -> 16
[609, 906] -> 7


### 6.4. Độ phức tạp Cách 2 và chỗ còn vướng

Gọi `L = len(high)` (tối đa 15) và `S` = tổng số strobogrammatic có độ dài trong `[len(low), len(high)]`. Ở mục 8 ta sẽ tính được `S <= 312 499`.

- **Thời gian:** `O(S · L)` — mỗi số được dựng bằng các phép nối chuỗi dài tới `L`, rồi so sánh chuỗi `O(L)`.
- **Bộ nhớ:** `O(S_max · L)` — phải **giữ cả list** cho một độ dài; riêng `L = 15` đã là **187 500** chuỗi.

Chạy được với giới hạn đề, nhưng có hai chỗ "phí":

1. Ta **lưu cả danh sách** chỉ để **đếm** ⇒ Cách 3 đếm thẳng, bộ nhớ chỉ `O(L)`.
2. Ở các độ dài nằm **hẳn giữa**, ta vẫn dựng từng số dù biết chắc **tất cả** đều được tính ⇒ Cách 4 dùng công thức.

## 7. Cách 3 (cách chính) — DFS đặt cặp từ ngoài vào trong, đếm trực tiếp

### 7.1. Ý tưởng

**DFS** (depth-first search, tìm kiếm theo chiều sâu) là đi **hết một nhánh** của cây lựa chọn tới tận lá, rồi quay lại (backtrack) thử nhánh khác. Với mỗi độ dài `n`:

- Tạo mảng `buf = [""] * n`. Hàm `dfs(left, right)` điền cặp vị trí `(left, right)`, bắt đầu từ `(0, n - 1)`.
- Ở mỗi bước, thử lần lượt 5 cặp `(a, b)` trong `PAIRS`, nhưng **bỏ qua**:
  - cặp `0-0` khi `left == 0` và `n > 1` (số 0 đứng đầu);
  - cặp có `a != b` khi `left == right` (ô tâm chỉ được `0/1/8`).
- Đặt `buf[left], buf[right] = a, b`, rồi gọi `dfs(left + 1, right - 1)` để điền vào trong.
- Khi `left > right`: đã điền đủ ⇒ một **lá** = một số hoàn chỉnh. So với `low`/`high` (chỉ khi cùng độ dài), trả về `1` nếu nằm trong khoảng, `0` nếu không.
- `dfs` trả về **số lá hợp lệ** bên dưới nó; cộng dồn qua các lựa chọn.

Không cần "dọn" `buf` sau khi quay lui: lựa chọn kế tiếp **ghi đè** đúng hai ô đó.

Cây lựa chọn với `n = 3`:

```text
(0,2) cặp ngoài ∈ {1-1, 6-9, 8-8, 9-6}       (không có 0-0: số 0 đứng đầu)
 ├─ 1_1 -> (1,1) tâm ∈ {0, 1, 8} -> 101, 111, 181
 ├─ 6_9 ->                          609, 619, 689
 ├─ 8_8 ->                          808, 818, 888
 └─ 9_6 ->                          906, 916, 986
```

In [16]:
def count_length_dfs(n: int, low: str, high: str) -> int:
    """Đếm số strobogrammatic dài n nằm trong [low, high]."""
    buf = [""] * n

    def dfs(left: int, right: int) -> int:
        if left > right:                                # lá: đã có số hoàn chỉnh
            s = "".join(buf)
            if n == len(low) and s < low:
                return 0
            if n == len(high) and s > high:
                return 0
            return 1
        total = 0
        for a, b in PAIRS:
            if left == right and a != b:                # ô tâm: chỉ 0, 1, 8
                continue
            if left == 0 and n > 1 and a == "0":        # cấm số 0 đứng đầu
                continue
            buf[left], buf[right] = a, b
            total += dfs(left + 1, right - 1)
        return total

    return dfs(0, n - 1)


def strobo_range_dfs(low: str, high: str) -> int:
    return sum(count_length_dfs(n, low, high) for n in range(len(low), len(high) + 1))


for low, high in [("50", "100"), ("0", "0"), ("0", "100"), ("10", "1000"), ("609", "906")]:
    got = strobo_range_dfs(low, high)
    assert got == strobo_range_bruteforce(low, high)
    print(f"[{low}, {high}] -> {got}")

[50, 100] -> 3
[0, 0] -> 1
[0, 100] -> 7
[10, 1000] -> 16
[609, 906] -> 7


### 7.2. Trace Example 1: `low = "50"`, `high = "100"`

Độ dài xét: `n = 2` (bằng `len(low)` ⇒ chỉ so với `low`) và `n = 3` (bằng `len(high)` ⇒ chỉ so với `high`).

| n | Lá (số hoàn chỉnh) | So sánh | Quyết định | count |
|:-:|---|---|---|:-:|
| 2 | `11` | `"11" < "50"` | bỏ | 0 |
| 2 | `69`, `88`, `96` | `>= "50"` | **đếm** | 3 |
| 3 | `101` | `"101" > "100"` | bỏ | 3 |
| 3 | 11 số còn lại (`111` ... `986`) | đều `> "100"` | bỏ | 3 |

Chú ý cặp `0-0` không bao giờ xuất hiện ở ngoài cùng, và ở `n = 3` tâm chỉ thử `0, 1, 8`. Hàm `trace_dfs` in từng lá:

In [17]:
def trace_dfs(low: str, high: str) -> int:
    count = 0
    for n in range(len(low), len(high) + 1):
        checks = [name for name, ok in (("low", n == len(low)), ("high", n == len(high))) if ok]
        print(f"--- n = {n}: so với {', '.join(checks) or 'không gì (độ dài nằm giữa)'} ---")
        buf = [""] * n

        def dfs(left, right):
            nonlocal count                      # cho phép hàm trong sửa biến count của hàm ngoài
            if left > right:
                s = "".join(buf)
                if n == len(low) and s < low:
                    print(f"  lá {s}: {s!r} < low {low!r} -> bỏ")
                elif n == len(high) and s > high:
                    print(f"  lá {s}: {s!r} > high {high!r} -> bỏ")
                else:
                    count += 1
                    print(f"  lá {s}: trong khoảng -> ĐẾM, count = {count}")
                return
            for a, b in PAIRS:
                if left == right and a != b:
                    continue
                if left == 0 and n > 1 and a == "0":
                    continue
                buf[left], buf[right] = a, b
                dfs(left + 1, right - 1)

        dfs(0, n - 1)
    print("Kết quả:", count)
    return count


_ = trace_dfs("50", "100")

--- n = 2: so với low ---
  lá 11: '11' < low '50' -> bỏ
  lá 69: trong khoảng -> ĐẾM, count = 1
  lá 88: trong khoảng -> ĐẾM, count = 2
  lá 96: trong khoảng -> ĐẾM, count = 3
--- n = 3: so với high ---
  lá 101: '101' > high '100' -> bỏ
  lá 111: '111' > high '100' -> bỏ
  lá 181: '181' > high '100' -> bỏ
  lá 609: '609' > high '100' -> bỏ
  lá 619: '619' > high '100' -> bỏ
  lá 689: '689' > high '100' -> bỏ
  lá 808: '808' > high '100' -> bỏ
  lá 818: '818' > high '100' -> bỏ
  lá 888: '888' > high '100' -> bỏ
  lá 906: '906' > high '100' -> bỏ
  lá 916: '916' > high '100' -> bỏ
  lá 986: '986' > high '100' -> bỏ
Kết quả: 3


### 7.3. Độ phức tạp Cách 3

Với cùng ký hiệu `S` (số lá phải thăm) và `L = len(high)`:

- **Thời gian:** `O(S · L)`. Mỗi lá tốn `O(L)` cho `"".join(buf)` và phép so sánh. Số nút **trong** của cây không vượt quá số lá (mỗi nút trong có ít nhất 3 nhánh con), nên không làm tăng bậc.
- **Bộ nhớ:** `O(L)` — chỉ một mảng `buf` dài `L` và call stack sâu khoảng `L / 2`. **Không lưu danh sách** như Cách 2.

Cách 3 là lời giải "chuẩn phỏng vấn": ngắn, dùng lại đúng tư duy của 246/247, xử lý số 0 đứng đầu và ô tâm ngay tại chỗ, thừa sức chạy giới hạn đề (tối đa ≈ 312 nghìn lá).

## 8. Cách 4 — Đếm bằng công thức cho các độ dài nằm hẳn giữa

### 8.1. Có bao nhiêu số strobogrammatic dài đúng `n`?

Gọi số đó là `f(n)`. Dùng quy tắc nhân (mục 3.7): chỉ cần **chọn nửa đầu**, nửa sau bị buộc theo.

- `n = 1`: chỉ có `0, 1, 8` ⇒ `f(1) = 3`.
- `n = 2`: cặp ngoài `1-1, 6-9, 8-8, 9-6` (cấm `0-0`) ⇒ `f(2) = 4`.
- `n = 3`: cặp ngoài 4 cách × tâm 3 cách ⇒ `f(3) = 12`.
- `n = 4`: cặp ngoài 4 cách × cặp trong 5 cách (có `0-0`) ⇒ `f(4) = 20`.
- `n = 5`: `4 × 5 × 3 = 60`.

Tổng quát với `n >= 2`: có **1 cặp ngoài** (4 cách), **`(n - 2) // 2` cặp trong** (mỗi cặp 5 cách), và nếu `n` lẻ thêm **1 ô tâm** (3 cách):

```text
f(n) = 4 × 5^((n - 2) // 2) × (3 nếu n lẻ, 1 nếu n chẵn)       (n >= 2),   f(1) = 3
```

Kiểm tra `(n - 2) // 2` với `n = 5`: các cặp là `(0,4), (1,3)` ⇒ 1 cặp trong ✓ và `(5 - 2) // 2 = 1` ✓. Ô dưới so công thức với số lượng sinh thật bằng `find_strobogrammatic`.

In [18]:
def count_full_length(n: int) -> int:
    """f(n): số strobogrammatic có đúng n chữ số (n >= 1)."""
    if n == 1:
        return 3
    return 4 * 5 ** ((n - 2) // 2) * (3 if n % 2 == 1 else 1)


print(" n | f(n) công thức | sinh thật")
for n in range(1, 16):
    real = len(find_strobogrammatic(n)) if n <= 12 else "-"
    if n <= 12:
        assert real == count_full_length(n)
    print(f"{n:>2} | {count_full_length(n):>14,} | {real}")
print("Tổng n = 1..15:", sum(count_full_length(n) for n in range(1, 16)))

 n | f(n) công thức | sinh thật
 1 |              3 | 3
 2 |              4 | 4
 3 |             12 | 12
 4 |             20 | 20
 5 |             60 | 60
 6 |            100 | 100
 7 |            300 | 300
 8 |            500 | 500
 9 |          1,500 | 1500
10 |          2,500 | 2500
11 |          7,500 | 7500
12 |         12,500 | 12500
13 |         37,500 | -
14 |         62,500 | -
15 |        187,500 | -
Tổng n = 1..15: 312499


### 8.2. Thuật toán Cách 4

Chỉ hai độ dài **biên** (`len(low)` và `len(high)`) mới cần so sánh từng số. Mọi độ dài `L` với `len(low) < L < len(high)` thì **toàn bộ** `f(L)` số đều nằm trong khoảng ⇒ cộng thẳng `f(L)`, không cần sinh.

So với Cách 3, code chỉ đổi đúng một chỗ: độ dài nằm giữa thì dùng `count_full_length` thay cho `count_length_dfs`.

Ví dụ `low = "50"`, `high = "123456"`: độ dài 2 (biên dưới) chạy DFS; độ dài 3, 4, 5 cộng `12 + 20 + 60 = 92`; độ dài 6 (biên trên) chạy DFS.

In [19]:
def strobo_range_formula(low: str, high: str) -> int:
    total = 0
    for n in range(len(low), len(high) + 1):
        if n == len(low) or n == len(high):
            total += count_length_dfs(n, low, high)    # độ dài biên: phải so từng số
        else:
            total += count_full_length(n)              # độ dài nằm giữa: công thức
    return total


for low, high in [("50", "100"), ("0", "0"), ("10", "1000"), ("50", "123456")]:
    got = strobo_range_formula(low, high)
    assert got == strobo_range_dfs(low, high)
    print(f"[{low}, {high}] -> {got}")

[50, 100] -> 3
[0, 0] -> 1
[10, 1000] -> 16
[50, 123456] -> 105


### 8.3. Cách 4 nhanh hơn bao nhiêu — và còn vướng ở đâu?

Đếm số **lá** mỗi cách phải thăm (con số chắc chắn, không phụ thuộc máy) và đo thời gian (phụ thuộc máy):

- Khoảng `["0", "999999999999999"]`: Cách 3 thăm mọi lá độ dài 1..15; Cách 4 chỉ thăm lá độ dài 1 và 15.
- Khoảng `["100000000000000", "999999999999999"]`: cả hai biên **cùng** độ dài 15 ⇒ Cách 4 **không** tiết kiệm được gì, vẫn phải dựng 187 500 số.

In [20]:
def leaves_dfs(low, high):          # số lá Cách 3 phải thăm
    return sum(count_full_length(n) for n in range(len(low), len(high) + 1))


def leaves_formula(low, high):      # Cách 4 chỉ thăm lá ở độ dài biên
    return sum(count_full_length(n) for n in {len(low), len(high)})


for low, high in [("0", "999999999999999"), ("100000000000000", "999999999999999")]:
    print(f"Khoảng [{low}, {high}]")
    for name, fn, leaves in [("Cách 3", strobo_range_dfs, leaves_dfs), ("Cách 4", strobo_range_formula, leaves_formula)]:
        t0 = time.perf_counter()
        ans = fn(low, high)
        dt = time.perf_counter() - t0
        print(f"  {name}: đáp án {ans:,} | thăm {leaves(low, high):,} lá | {dt:.3f} giây")

Khoảng [0, 999999999999999]


  Cách 3: đáp án 312,499 | thăm 312,499 lá | 0.100 giây


  Cách 4: đáp án 312,499 | thăm 187,503 lá | 0.058 giây
Khoảng [100000000000000, 999999999999999]


  Cách 3: đáp án 187,500 | thăm 187,500 lá | 0.062 giây
  Cách 4: đáp án 187,500 | thăm 187,500 lá | 0.060 giây


Kết luận: Cách 4 giúp khi hai biên khác độ dài, nhưng ở **độ dài biên** vẫn phải liệt kê từng số. Với đề này (tối đa 15 chữ số) như thế là đủ nhanh. Nhưng thử tưởng tượng đề cho 30 chữ số: `f(30) = 4 × 5^14 ≈ 2,4 × 10^10` lá — không thể liệt kê.

Câu hỏi tự nhiên: ở độ dài biên, có cách **đếm** số strobogrammatic `<= high` mà **không** dựng từng số không? Đó là Cách 5.

## 9. Cách 5 (nâng cao) — Đếm từng chữ số với `F(x)`

### 9.1. Kỹ thuật "đếm tới": `F(high) - F(low) + [low hợp lệ]`

Đặt `F(x)` = số strobogrammatic trong đoạn `[0, x]`. Khi đó:

```text
đáp án = F(high) - F(low) + (1 nếu low strobogrammatic, ngược lại 0)
```

Vì sao? `F(high)` đếm `[0, high]`, `F(low)` đếm `[0, low]`; hiệu là số trong `(low, high]` — **mất** đúng số `low`. Nếu `low` hợp lệ thì cộng lại 1. (Cách quen thuộc hơn là `F(high) - F(low - 1)`, nhưng tính `low - 1` trên chuỗi khá phiền: `"100"` thành `"99"` đổi độ dài, `"0"` thành `-1`... nên ta cộng bù như trên.)

Ví dụ mini với số chẵn: số chẵn trong `[4, 10]` = `F(10) - F(4) + [4 chẵn]` với `F(10) = |{0,2,4,6,8,10}| = 6`, `F(4) = |{0,2,4}| = 3` ⇒ `6 - 3 + 1 = 4` (đúng: `4, 6, 8, 10`).

In [21]:
def F_even(x):                               # số chẵn trong [0, x], đếm "thô"
    return sum(1 for v in range(x + 1) if v % 2 == 0)


low_v, high_v = 4, 10
print(F_even(high_v) - F_even(low_v) + (1 if low_v % 2 == 0 else 0))   # 4

F_strobo_brute = lambda x: strobo_range_bruteforce("0", x)              # F(x) bản brute force
print(F_strobo_brute("100") - F_strobo_brute("50") + is_strobogrammatic("50"))   # 3 = Example 1
print("True + 0 =", True + 0, "-> trong Python, bool cộng như số 0/1")

4
3
True + 0 = 1 -> trong Python, bool cộng như số 0/1


### 9.2. Bài toán con: đếm số có độ dài cố định, chữ số trong tập cho phép, `<= bound`

Trước khi đếm strobogrammatic, luyện một bài đơn giản hơn: **có bao nhiêu số dài 3, mỗi chữ số thuộc `{1, 3, 5}`, mà `<= 352`?**

Ý tưởng then chốt — **vị trí khác nhau đầu tiên**: lấy một số `x < bound` cùng độ dài, xét vị trí `i` đầu tiên (từ trái) mà `x[i] != bound[i]`. Vì `x < bound` nên `x[i] < bound[i]`, và `x[0..i-1]` **trùng** `bound[0..i-1]`. Các chữ số **sau** `i` thì tùy ý. Ta chia mọi số `x` thành các nhóm theo `i`:

| i | Điều kiện tiền tố | `x[i]` được chọn (`< bound[i]`, trong tập) | Phần sau tự do | Đóng góp |
|:-:|---|---|---|:-:|
| 0 | — | `< '3'`: `{1}` → 1 cách | 2 ô × 3 cách = 9 | `1 × 9 = 9` |
| 1 | `x[0] = '3'` (hợp lệ ✓) | `< '5'`: `{1, 3}` → 2 cách | 1 ô = 3 | `2 × 3 = 6` |
| 2 | `x[0..1] = "35"` (hợp lệ ✓) | `< '2'`: `{1}` → 1 cách | 0 ô = 1 | `1 × 1 = 1` |
| — | `x = bound = "352"`? | chữ `'2'` không thuộc tập ⇒ không tính | | 0 |

Tổng: `9 + 6 + 1 = 16`. Quy tắc dừng: nếu `bound[i]` **không** thuộc tập cho phép thì không số `x` nào trùng được tiền tố dài hơn ⇒ **dừng** ngay. Nếu đi hết mà mọi `bound[i]` đều hợp lệ thì chính `bound` cũng được đếm (`+1`).

(Đây cũng chính là lõi của LeetCode 902 — *Numbers At Most N Given Digit Set*, một bài Hard khác.)

In [22]:
def count_fixed_length_le(bound: str, allowed: str) -> int:
    n = len(bound)
    total = 0
    for i in range(n):
        smaller = sum(1 for d in allowed if d < bound[i])
        total += smaller * len(allowed) ** (n - 1 - i)   # x[i] nhỏ hơn, phần sau tự do
        if bound[i] not in allowed:                      # không thể trùng tiền tố dài hơn
            return total
    return total + 1                                     # chính bound cũng hợp lệ


for bound in ["352", "355", "100", "999", "531"]:
    brute = sum(1 for t in product("135", repeat=3) if "".join(t) <= bound)   # cùng độ dài -> so chuỗi an toàn
    got = count_fixed_length_le(bound, "135")
    assert got == brute
    print(f"bound = {bound}: {got} (brute force: {brute})")

bound = 352: 16 (brute force: 16)
bound = 355: 18 (brute force: 18)
bound = 100: 0 (brute force: 0)
bound = 999: 27 (brute force: 27)
bound = 531: 22 (brute force: 22)


### 9.3. Áp dụng cho strobogrammatic: chỉ **nửa đầu** là tự do

Một số strobogrammatic dài `n` được quyết định hoàn toàn bởi **nửa đầu**: vị trí `0 .. half - 1` với `half = (n + 1) // 2` (gồm cả ô tâm nếu `n` lẻ). Nửa sau là bản xoay của nửa đầu. Tập chữ số cho phép ở vị trí `i` của nửa đầu:

| Vị trí | Tập cho phép | Số cách |
|---|---|:-:|
| ô tâm (`n` lẻ, `i = n // 2`) — kể cả khi `n = 1` | `0, 1, 8` | 3 |
| `i = 0` (đứng đầu, `n > 1`) | `1, 6, 8, 9` | 4 |
| các vị trí khác | `0, 1, 6, 8, 9` | 5 |

Đếm số strobogrammatic `x` dài `n` mà `x <= bound` (cùng độ dài). Có hai loại `x`:

- **Loại A — `x` khác `bound` ở đâu đó trong nửa đầu.** Gọi `i` là vị trí khác đầu tiên. Phép so sánh `x` với `bound` được quyết định **ngay tại `i`** (nửa sau không còn ảnh hưởng), nên `x < bound` ⇔ `x[i] < bound[i]`. Đếm y hệt mục 9.2: `(số chữ cho phép < bound[i]) × (tích số cách của các vị trí nửa đầu sau i)`, và dừng khi `bound[i]` không thuộc tập.
- **Loại B — nửa đầu của `x` trùng hoàn toàn nửa đầu của `bound`.** Khi đó `x` được xác định **duy nhất**: `x = bound[:half] + rotate(bound[:n // 2])`. Chỉ cần so một lần `x <= bound` để quyết định `+1` hay `+0`. Đây là điểm khác với mục 9.2: ở đó ta `+1` luôn, còn ở đây nửa sau của `x` có thể **lớn hơn** nửa sau của `bound`.

Cuối cùng: `F(bound) = f(1) + f(2) + ... + f(len(bound) - 1)` (mọi số ngắn hơn, có cả số `0`) `+` số cùng độ dài `<= bound`.

In [23]:
def allowed_digits(i: int, n: int) -> list:
    """Chữ số được đặt ở vị trí i (trong nửa đầu) của số strobogrammatic dài n."""
    if n % 2 == 1 and i == n // 2:           # ô tâm (kể cả n == 1: số 0 hợp lệ)
        return ["0", "1", "8"]
    if i == 0:                               # đứng đầu: cấm 0
        return ["1", "6", "8", "9"]
    return ["0", "1", "6", "8", "9"]


def count_same_length_le(bound: str) -> int:
    n = len(bound)
    half = (n + 1) // 2
    choices = [allowed_digits(i, n) for i in range(half)]
    total = 0
    for i in range(half):                                # Loại A
        rest = 1
        for c in choices[i + 1:]:
            rest *= len(c)
        smaller = sum(1 for d in choices[i] if d < bound[i])
        total += smaller * rest
        if bound[i] not in choices[i]:
            return total
    candidate = bound[:half] + rotate(bound[: n // 2])   # Loại B: ứng viên duy nhất
    return total + (1 if candidate <= bound else 0)


def count_le(bound: str) -> int:                         # F(bound)
    shorter = sum(count_full_length(k) for k in range(1, len(bound)))
    return shorter + count_same_length_le(bound)


def strobo_range_math(low: str, high: str) -> int:
    return count_le(high) - count_le(low) + (1 if is_strobogrammatic(low) else 0)


for low, high in [("50", "100"), ("0", "0"), ("0", "100"), ("10", "1000"), ("609", "906")]:
    got = strobo_range_math(low, high)
    assert got == strobo_range_bruteforce(low, high)
    print(f"[{low}, {high}] -> {got}")

[50, 100] -> 3
[0, 0] -> 1
[0, 100] -> 7
[10, 1000] -> 16
[609, 906] -> 7


### 9.4. Trace `F("8200")` và `F("6970")`

Cả hai bound dài 4 ⇒ phần độ dài ngắn hơn đóng góp `f(1) + f(2) + f(3) = 3 + 4 + 12 = 19`; `half = 2`, tập cho phép `[1689, 01689]`, tích phần còn lại sau `i = 0` là 5, sau `i = 1` là 1.

**`bound = "8200"`** — dừng giữa chừng vì gặp chữ `2`:

| i | Tập cho phép | bound[i] | Chữ nhỏ hơn | rest | Cộng | total | bound[i] hợp lệ? |
|:-:|---|:-:|---|:-:|:-:|:-:|---|
| 0 | `1 6 8 9` | 8 | `1, 6` | 5 | 10 | 10 | có → đi tiếp |
| 1 | `0 1 6 8 9` | 2 | `0, 1` | 1 | 2 | 12 | **không** → dừng |

Cùng độ dài: 12 (đó là `1001 1111 1691 1881 1961 6009 6119 6699 6889 6969 8008 8118`) ⇒ `F("8200") = 19 + 12 = 31`.

**`bound = "6970"`** — đi hết nửa đầu, rơi vào Loại B:

| i | Tập cho phép | bound[i] | Chữ nhỏ hơn | rest | Cộng | total | bound[i] hợp lệ? |
|:-:|---|:-:|---|:-:|:-:|:-:|---|
| 0 | `1 6 8 9` | 6 | `1` | 5 | 5 | 5 | có → đi tiếp |
| 1 | `0 1 6 8 9` | 9 | `0, 1, 6, 8` | 1 | 4 | 9 | có → đi tiếp |
| B | ứng viên `"69" + rotate("69")` = `"6969"` | | `"6969" <= "6970"` | | 1 | 10 | |

⇒ `F("6970") = 19 + 10 = 29`.

In [24]:
def trace_count_le(bound: str) -> int:
    n, half = len(bound), (len(bound) + 1) // 2
    shorter = sum(count_full_length(k) for k in range(1, n))
    print(f"bound = {bound!r}: n = {n}, half = {half}, độ dài ngắn hơn đóng góp {shorter}")
    choices = [allowed_digits(i, n) for i in range(half)]
    total = 0
    for i in range(half):
        rest = 1
        for c in choices[i + 1:]:
            rest *= len(c)
        smaller = [d for d in choices[i] if d < bound[i]]
        total += len(smaller) * rest
        ok = bound[i] in choices[i]
        print(f"  i={i}: cho phép={''.join(choices[i]):<5} bound[i]={bound[i]} nhỏ hơn={''.join(smaller) or '-':<4} "
              f"rest={rest} cộng {len(smaller) * rest:<2} total={total:<2} -> {'đi tiếp' if ok else 'DỪNG'}")
        if not ok:
            break
    else:                                    # vòng for chạy hết, không gặp break -> Loại B
        candidate = bound[:half] + rotate(bound[: n // 2])
        plus = 1 if candidate <= bound else 0
        total += plus
        print(f"  Loại B: ứng viên {candidate!r} {'<=' if plus else '>'} {bound!r} -> cộng {plus}, total={total}")
    print(f"  => F({bound!r}) = {shorter} + {total} = {shorter + total}")
    return shorter + total


assert trace_count_le("8200") == count_le("8200") == F_strobo_brute("8200")
assert trace_count_le("6970") == count_le("6970") == F_strobo_brute("6970")

bound = '8200': n = 4, half = 2, độ dài ngắn hơn đóng góp 19
  i=0: cho phép=1689  bound[i]=8 nhỏ hơn=16   rest=5 cộng 10 total=10 -> đi tiếp
  i=1: cho phép=01689 bound[i]=2 nhỏ hơn=01   rest=1 cộng 2  total=12 -> DỪNG
  => F('8200') = 19 + 12 = 31
bound = '6970': n = 4, half = 2, độ dài ngắn hơn đóng góp 19
  i=0: cho phép=1689  bound[i]=6 nhỏ hơn=1    rest=5 cộng 5  total=5  -> đi tiếp
  i=1: cho phép=01689 bound[i]=9 nhỏ hơn=0168 rest=1 cộng 4  total=9  -> đi tiếp
  Loại B: ứng viên '6969' <= '6970' -> cộng 1, total=10
  => F('6970') = 19 + 10 = 29


Ở `trace_count_le` có cú pháp mới `for ... else`: khối `else` của vòng `for` chạy **chỉ khi** vòng lặp đi hết mà **không** gặp `break`. Đúng ý "đi hết nửa đầu ⇒ xét Loại B". (Bản `count_same_length_le` dùng `return` sớm nên không cần `else`.)

### 9.5. Độ phức tạp Cách 5

Gọi `L = len(high)`.

- **Thời gian:** `O(L²)` — mỗi `count_le` duyệt `half ≈ L/2` vị trí, mỗi vị trí tính `rest` bằng một vòng `O(L)` và so tối đa 5 chữ số; cộng `f(k)` cho `k < L` cũng `O(L)` lần. (Có thể hạ xuống `O(L)` bằng tích hậu tố, nhưng với `L <= 15` không cần.) **Không phụ thuộc** số lượng số strobogrammatic!
- **Bộ nhớ:** `O(L)`.

Ô dưới chứng minh sức mạnh đó: `F` của số **40 chữ số** tính ngay lập tức, khớp với tổng công thức `f(1) + ... + f(40)` — điều mà Cách 2, 3, 4 không bao giờ làm nổi.

In [25]:
big = "9" * 40
t0 = time.perf_counter()
value = count_le(big)
dt = time.perf_counter() - t0
assert value == sum(count_full_length(k) for k in range(1, 41))
print(f"F('9' * 40) = {value:,}  (tính trong {dt * 1000:.2f} ms)")
print("Đếm trong [10^20, 10^30]:", f"{strobo_range_math('1' + '0' * 20, '1' + '0' * 30):,}")

F('9' * 40) = 152,587,890,624,999  (tính trong 0.16 ms)
Đếm trong [10^20, 10^30]: 48,812,500,000


## 10. Code nộp LeetCode

Bản nộp chính dùng **Cách 3** (DFS từ ngoài vào trong): ngắn, dễ giải thích, dùng lại đúng tư duy 246/247, chạy dư sức với 15 chữ số. Bản `SolutionMath` (Cách 5, `O(L²)`) để dành khi người phỏng vấn hỏi "nếu độ dài lên 100 thì sao?". Trong notebook đặt tên class khác nhau để không đè nhau; **khi nộp, đổi tên thành `Solution`**.

Lưu ý trong `SolutionMath`: `sum(d < bound[i] for d in ...)` đếm số lần điều kiện đúng (vì `True` cộng như `1`), và `rot.get(c)` trả `None` với chữ số không xoay được.

In [26]:
class Solution:
    def strobogrammaticInRange(self, low: str, high: str) -> int:
        pairs = [("0", "0"), ("1", "1"), ("6", "9"), ("8", "8"), ("9", "6")]
        total = 0
        for n in range(len(low), len(high) + 1):
            buf = [""] * n

            def dfs(left: int, right: int) -> int:
                if left > right:
                    s = "".join(buf)
                    if n == len(low) and s < low:
                        return 0
                    if n == len(high) and s > high:
                        return 0
                    return 1
                count = 0
                for a, b in pairs:
                    if left == right and a != b:          # ô tâm: 0, 1, 8
                        continue
                    if left == 0 and n > 1 and a == "0":  # không có số 0 đứng đầu
                        continue
                    buf[left], buf[right] = a, b
                    count += dfs(left + 1, right - 1)
                return count

            total += dfs(0, n - 1)
        return total


class SolutionMath:
    def strobogrammaticInRange(self, low: str, high: str) -> int:
        rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}

        def full(n: int) -> int:                 # số strobogrammatic có đúng n chữ số
            if n == 1:
                return 3
            return 4 * 5 ** ((n - 2) // 2) * (3 if n % 2 else 1)

        def allowed(i: int, n: int) -> str:
            if n % 2 == 1 and i == n // 2:
                return "018"
            return "1689" if i == 0 else "01689"

        def count_le(bound: str) -> int:         # số strobogrammatic trong [0, bound]
            n = len(bound)
            half = (n + 1) // 2
            total = sum(full(k) for k in range(1, n))
            for i in range(half):
                rest = 1
                for j in range(i + 1, half):
                    rest *= len(allowed(j, n))
                total += sum(d < bound[i] for d in allowed(i, n)) * rest
                if bound[i] not in allowed(i, n):
                    return total
            mirror = "".join(rot[c] for c in reversed(bound[: n // 2]))
            return total + (bound[:half] + mirror <= bound)

        low_ok = all(rot.get(c) == low[-1 - i] for i, c in enumerate(low))
        return count_le(high) - count_le(low) + low_ok

### Bộ test

1. Hai ví dụ của đề.
2. Khoảng chỉ có 1 số: số đó hợp lệ (`"69"`, `"0"`) và không hợp lệ (`"962"`).
3. Khoảng một chữ số: `["0","9"]` (có `0, 1, 8`), `["2","5"]` (không có gì), `["6","9"]` (chỉ `8` — nhớ `6`, `9` đứng một mình không hợp lệ).
4. Biên rơi đúng vào số strobogrammatic: `["609","906"]` ⇒ 7 (`609 619 689 808 818 888 906`).
5. Độ dài nằm giữa: `["10","1000"]` ⇒ 16; trọn một độ dài `["1000","9999"]` ⇒ `f(4) = 20`.
6. Giới hạn đề: `["0", "999999999999999"]` ⇒ 312 499; `["100000000000000", "999999999999999"]` ⇒ 187 500; `["1", "100000000000000"]` ⇒ 124 998.

### Test đối chiếu chéo trên input ngẫu nhiên

Sinh 2000 khoảng `[low, high]` ngẫu nhiên (`random.seed(248)` cố định để lần nào chạy cũng giống nhau). Để bắt được các nhánh "biên trùng đúng số strobogrammatic", khoảng 30% lần ta chọn `low` hoặc `high` **chính là** một số strobogrammatic. Brute force (Cách 1) làm **trọng tài**, nên giữ khoảng dài tối đa vài nghìn số. Sau đó so riêng 4 cách nhanh với nhau trên các khoảng **rất rộng** (tới 10 chữ số) mà brute force không chạy nổi.

In [27]:
def check(name, low, high, expected):
    got = [Solution().strobogrammaticInRange(low, high), SolutionMath().strobogrammaticInRange(low, high)]
    assert got == [expected, expected], (name, low, high, got, expected)
    print(f"{name:<22} [{low}, {high}] -> {expected}  PASS")


check("example 1", "50", "100", 3)
check("example 2", "0", "0", 1)
check("1 số, hợp lệ", "69", "69", 1)
check("1 số, không hợp lệ", "962", "962", 0)
check("một chữ số", "0", "9", 3)
check("không có gì", "2", "5", 0)
check("6, 9 đứng một mình", "6", "9", 1)
check("biên là strobo", "609", "906", 7)
check("độ dài nằm giữa", "10", "1000", 16)
check("trọn độ dài 4", "1000", "9999", 20)
check("giới hạn: 1..15 chữ số", "0", "999999999999999", 312499)
check("giới hạn: 15 chữ số", "100000000000000", "999999999999999", 187500)
check("giới hạn: 10^14", "1", "100000000000000", 124998)
print("Tất cả test đều PASS!")


import random

random.seed(248)
strobo_pool = [int(s) for n in range(1, 7) for s in find_strobogrammatic(n)]
fast_methods = [strobo_range_generate, strobo_range_dfs, strobo_range_formula, strobo_range_math,
                Solution().strobogrammaticInRange, SolutionMath().strobogrammaticInRange]

for _ in range(2000):
    lo = random.choice(strobo_pool) if random.random() < 0.3 else random.randint(0, 10 ** random.randint(1, 6))
    hi = lo + random.randint(0, 3000)
    if random.random() < 0.3:
        bigger = [v for v in strobo_pool if v >= lo][:15]
        if bigger:
            hi = random.choice(bigger)
    low, high = str(lo), str(hi)
    expected = strobo_range_bruteforce(low, high)
    assert all(fn(low, high) == expected for fn in fast_methods), (low, high)
print("2000 khoảng nhỏ: 6 hàm nhanh khớp brute force.")

for _ in range(300):
    a = random.randint(0, 10 ** random.randint(1, 10))
    b = random.randint(0, 10 ** random.randint(1, 10))
    low, high = str(min(a, b)), str(max(a, b))
    answers = {fn(low, high) for fn in fast_methods}
    assert len(answers) == 1, (low, high, answers)
print("300 khoảng rộng (tới 10 chữ số): 6 hàm nhanh cho cùng kết quả.")

example 1              [50, 100] -> 3  PASS
example 2              [0, 0] -> 1  PASS
1 số, hợp lệ           [69, 69] -> 1  PASS
1 số, không hợp lệ     [962, 962] -> 0  PASS
một chữ số             [0, 9] -> 3  PASS
không có gì            [2, 5] -> 0  PASS
6, 9 đứng một mình     [6, 9] -> 1  PASS
biên là strobo         [609, 906] -> 7  PASS
độ dài nằm giữa        [10, 1000] -> 16  PASS
trọn độ dài 4          [1000, 9999] -> 20  PASS


giới hạn: 1..15 chữ số [0, 999999999999999] -> 312499  PASS


giới hạn: 15 chữ số    [100000000000000, 999999999999999] -> 187500  PASS


giới hạn: 10^14        [1, 100000000000000] -> 124998  PASS
Tất cả test đều PASS!


2000 khoảng nhỏ: 6 hàm nhanh khớp brute force.


300 khoảng rộng (tới 10 chữ số): 6 hàm nhanh cho cùng kết quả.


## 11. Vì sao đúng?

### Cách 3 (DFS) — mỗi số hợp lệ ứng với đúng một lá

Xét một độ dài `n` cố định.

- **Không thiếu:** mọi số strobogrammatic `x` dài `n` (không có số 0 đứng đầu) đều được dựng ra. Vì `x` strobogrammatic nên với mỗi cặp vị trí `(i, n-1-i)`, `(x[i], x[n-1-i])` là một cặp trong `PAIRS` (mục 3.3); cặp ngoài cùng không phải `0-0` (trừ `n = 1`); ô tâm có `x[i] = x[i]` nên qua được luật tâm. Vậy DFS có một đường đi chọn đúng các cặp đó, tới đúng lá `x`.
- **Không thừa:** mọi lá đều là một số strobogrammatic hợp lệ — vì mỗi cặp đặt vào là cặp xoay, ô tâm chỉ `0/1/8`, và chữ số đầu không phải `0` khi `n > 1`.
- **Không đếm trùng:** hai đường đi khác nhau chọn khác nhau ở ít nhất một cặp vị trí ⇒ hai lá là hai chuỗi khác nhau.
- **Lọc đúng khoảng:** số dài `n` thỏa `len(low) < n < len(high)` chắc chắn nằm trong khoảng; ở độ dài biên ta so **cùng độ dài** nên so chuỗi trùng so số (mục 3.4).

Cộng qua mọi độ dài (các nhóm rời nhau — quy tắc cộng) ra đúng đáp án.

### Cách 5 — chia `x <= bound` thành các nhóm rời nhau

Mọi số strobogrammatic `x` cùng độ dài với `bound` và `x <= bound` thuộc **đúng một** nhóm: hoặc vị trí khác nhau đầu tiên `i` nằm trong nửa đầu (khi đó `x[i] < bound[i]` và phần nửa đầu sau `i` tùy ý — đúng tích `rest`), hoặc nửa đầu trùng hoàn toàn (khi đó `x` là ứng viên duy nhất, so một lần). Các nhóm rời nhau ⇒ cộng lại bằng quy tắc cộng. Số ngắn hơn `bound` luôn nhỏ hơn `bound` (không có số 0 đứng đầu) ⇒ cộng `f(k)` cho mọi `k < len(bound)`.

## 12. So sánh các cách

Ký hiệu: `L = len(high) <= 15`; `S` = tổng số strobogrammatic có độ dài trong `[len(low), len(high)]` (tối đa 312 499); `R = int(high) - int(low) + 1` (số lượng số trong khoảng, tới `10^15`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | duyệt từng số + kiểm tra 246 | `O(R · L)` | `O(L)` | chỉ cho khoảng nhỏ; làm trọng tài |
| 2. Sinh theo độ dài (247) | đệ quy trong → ngoài, lọc | `O(S · L)` | `O(S · L)` | dễ hiểu, tốn bộ nhớ lưu list |
| 3. DFS ngoài → trong | điền cặp, đếm ở lá | `O(S · L)` | `O(L)` | **bản nộp chính**, ngắn, an toàn |
| 4. Công thức + DFS biên | `f(n)` cho độ dài nằm giữa | `O(S_biên · L)` | `O(L)` | nhanh hơn khi hai biên khác độ dài |
| 5. Đếm từng chữ số `F(x)` | `F(high) - F(low) + [low]` | `O(L²)` | `O(L)` | nhanh nhất, chạy được cả `L = 1000` |

Lời khuyên học tập: viết được Cách 3 một mạch là đủ để qua bài; hiểu Cách 5 thì bạn có thêm một "vũ khí" (đếm theo chữ số) dùng được cho rất nhiều bài Hard khác (902, 1088, 233, 2376...).

## 13. Lỗi thường gặp

### Lỗi 1 — So sánh chuỗi khác độ dài

Sai: sinh số ở mọi độ dài rồi lọc bằng `low <= s <= high` (so chuỗi):

```python
if low <= s <= high:     # "50" <= "69" đúng, nhưng "69" <= "100" SAI vì '6' > '1'
    count += 1
```

Với Example 1, các số `69, 88, 96` bị loại oan ⇒ ra `0` thay vì `3`. Sửa: chỉ so chuỗi ở **độ dài biên** (cùng độ dài), hoặc so bằng `(len(s), s)`.

In [28]:
def wrong_string_filter(low, high):
    return sum(1 for L in range(len(low), len(high) + 1) for s in find_strobogrammatic(L) if low <= s <= high)


def right_key_filter(low, high):
    key = lambda s: (len(s), s)
    return sum(1 for L in range(len(low), len(high) + 1) for s in find_strobogrammatic(L)
               if key(low) <= key(s) <= key(high))


print("Sai :", wrong_string_filter("50", "100"))   # 0
print("Đúng:", right_key_filter("50", "100"))      # 3

Sai : 0
Đúng: 3


### Lỗi 2 — Quên cấm số 0 đứng đầu

Nếu cho phép cặp `0-0` ở ngoài cùng, ta đếm thêm các "số" `00`, `000`, `010`, `080`... Với `["0", "100"]`: đúng là 7, sai thành 11 (`00` ở độ dài 2; `000, 010, 080` ở độ dài 3 vì `"000" < "100"` khi so chuỗi cùng độ dài).

### Lỗi 3 — Ô tâm dùng cả `6` / `9`

Quên luật `left == right and a != b`. Khi `left == right`, lệnh `buf[left], buf[right] = a, b` gán cùng một ô hai lần, ô đó nhận `b`, nên cặp `6-9` sinh ra tâm `9` và cặp `9-6` sinh ra tâm `6` ⇒ đếm thêm `161, 191, 669, 699, 868, 898...` Với `["100", "999"]`: đúng 12, sai thành 20.

In [29]:
def dfs_with_bugs(low, high, forbid_leading_zero=True, check_center=True):
    total = 0
    for n in range(len(low), len(high) + 1):
        buf = [""] * n

        def dfs(left, right):
            if left > right:
                s = "".join(buf)
                return 0 if (n == len(low) and s < low) or (n == len(high) and s > high) else 1
            cnt = 0
            for a, b in PAIRS:
                if check_center and left == right and a != b:
                    continue
                if forbid_leading_zero and left == 0 and n > 1 and a == "0":
                    continue
                buf[left], buf[right] = a, b
                cnt += dfs(left + 1, right - 1)
            return cnt

        total += dfs(0, n - 1)
    return total


print("Lỗi 2 ['0','100']  : sai", dfs_with_bugs("0", "100", forbid_leading_zero=False), "| đúng", strobo_range_dfs("0", "100"))
print("Lỗi 3 ['100','999']: sai", dfs_with_bugs("100", "999", check_center=False), "| đúng", strobo_range_dfs("100", "999"))

Lỗi 2 ['0','100']  : sai 11 | đúng 7
Lỗi 3 ['100','999']: sai 20 | đúng 12


### Lỗi 4 — Cấm cặp `0-0` ở **mọi** lớp (LeetCode 247)

Ngược với Lỗi 2: sợ số 0 đứng đầu nên bỏ hẳn cặp `0-0` khỏi `PAIRS`. Kết quả là mất những số có `0-0` ở lớp trong như `1001`, `6009`, `8008`, `9006` ⇒ `f(4)` ra 16 thay vì 20. Sửa: chỉ cấm `0-0` ở **lớp ngoài cùng** (`k == n`).

### Lỗi 5 — Cách 5 quên một mảnh

- Quên `+ [low strobogrammatic]` ⇒ Example 2 `["0","0"]` ra `0` thay vì `1` (hiệu `F(high) - F(low)` đã trừ mất chính `low`).
- Quên **Loại B** (ứng viên khi nửa đầu trùng `bound`) ⇒ `F("6970")` mất số `6969`.
- Quên **dừng** khi `bound[i]` không thuộc tập ⇒ đếm cả những số có tiền tố không hợp lệ (và có thể gặp `KeyError` khi xoay chữ `2`).

In [30]:
PAIRS_NO_ZERO = [p for p in PAIRS if p != ("0", "0")]
wrong_4 = [a + mid + b for mid in [a + b for a, b in PAIRS_NO_ZERO] for a, b in PAIRS_NO_ZERO]
print("Lỗi 4: f(4) sai =", len(wrong_4), "| đúng =", len(find_strobogrammatic(4)))
print("       bị mất:", sorted(set(find_strobogrammatic(4)) - set(wrong_4)))

forgot_low = count_le("0") - count_le("0")
print("Lỗi 5a ['0','0']: quên +[low] ->", forgot_low, "| đúng ->", strobo_range_math("0", "0"))


def same_length_no_case_b(bound):            # cố tình bỏ Loại B
    n, half = len(bound), (len(bound) + 1) // 2
    total = 0
    for i in range(half):
        rest = 1
        for j in range(i + 1, half):
            rest *= len(allowed_digits(j, n))
        total += sum(1 for d in allowed_digits(i, n) if d < bound[i]) * rest
        if bound[i] not in allowed_digits(i, n):
            return total
    return total


print("Lỗi 5b same-length('6970'): quên Loại B ->", same_length_no_case_b("6970"), "| đúng ->", count_same_length_le("6970"))

Lỗi 4: f(4) sai = 16 | đúng = 20
       bị mất: ['1001', '6009', '8008', '9006']
Lỗi 5a ['0','0']: quên +[low] -> 0 | đúng -> 1
Lỗi 5b same-length('6970'): quên Loại B -> 9 | đúng -> 10


### Lỗi 6 — Nhầm "soi gương" với "xoay 180°"

Hai cách hiểu sai hay gặp:

- Quên **đọc ngược**, chỉ lật từng chữ: so `num` với `"".join(ROT[c] for c in num)`. Khi đó `"69"` lật thành `"96"` ≠ `"69"` ⇒ kết luận sai là không strobogrammatic.
- Coi `6 → 6`, `9 → 9` (như palindrome) ⇒ `"66"` bị coi là hợp lệ, trong khi xoay `66` ra `99`.

Nhớ câu: **xoay 180° = đọc ngược + lật `6 ↔ 9`**.

### Lỗi 7 — Off-by-one trong brute force

`range(int(low), int(high))` **không lấy** `high` ⇒ Example 2 `["0","0"]` ra `0`. Phải là `range(int(low), int(high) + 1)`.

## 14. Mẹo nhận dạng pattern

- **"Đếm số trong `[low, high]` thỏa một tính chất về chữ số", khoảng khổng lồ (`10^15`, `10^18`)** ⇒ không duyệt. Hai hướng: (a) nếu số thỏa mãn **rất hiếm** và có cấu trúc, **sinh thẳng** chúng (Cách 2, 3); (b) **đếm theo chữ số** với `F(x)` = số lượng trong `[0, x]` rồi lấy hiệu (Cách 5, còn gọi là *digit DP* khi cần thêm trạng thái).
- **Tính chất "đối xứng"** (palindrome, strobogrammatic, ...) ⇒ chỉ **nửa đầu** là tự do; xây theo cặp vị trí `(i, n-1-i)` từ ngoài vào trong hoặc từ trong ra ngoài; nhớ **ô tâm** khi độ dài lẻ.
- **Input là chuỗi số** ⇒ cẩn thận so sánh: cùng độ dài mới so chuỗi được; khác độ dài thì so `(len, s)`.
- **Có độ dài "nằm hẳn giữa"** ⇒ cộng thẳng bằng công thức tổ hợp, chỉ xử lý kỹ hai biên.
- **"Không có số 0 đứng đầu"** ⇒ chữ số đầu có tập cho phép riêng; số `0` một chữ số thường là ngoại lệ phải xét riêng.

## 15. Biến thể và bài liên quan

| Bài | Độ khó | Liên hệ |
|---|---|---|
| 246. Strobogrammatic Number | Easy (Premium) | kiểm tra một số — mục 3.3 (hai con trỏ) |
| 247. Strobogrammatic Number II | Medium (Premium) | sinh mọi số dài `n` — Cách 2 |
| 1056. Confusing Number | Easy (Premium) | xoay xong ra số **khác** — Bài tập 5 |
| 1088. Confusing Number II | Hard (Premium) | đếm confusing number trong `[1, n]` — Bài tập 6 |
| 902. Numbers At Most N Given Digit Set | Hard | đúng bài toán con ở mục 9.2 |
| 906. Super Palindromes | Hard | sinh palindrome (nửa đầu) rồi kiểm tra |
| 9. Palindrome Number | Easy | "đối xứng" kiểu soi gương thay vì xoay |
| 233, 2376 | Hard | đếm theo chữ số (digit DP) |

Thử chuyển giao kỹ thuật: **đếm palindrome trong `[low, high]`** chỉ cần đổi bảng cặp thành `(d, d)` và cho tâm là mọi chữ số. Toàn bộ khung DFS của Cách 3 giữ nguyên:

In [31]:
def count_palindromes_in_range(low: str, high: str) -> int:
    pal_pairs = [(d, d) for d in "0123456789"]          # cặp "soi gương": giống hệt nhau
    total = 0
    for n in range(len(low), len(high) + 1):
        buf = [""] * n

        def dfs(left, right):
            if left > right:
                s = "".join(buf)
                return 0 if (n == len(low) and s < low) or (n == len(high) and s > high) else 1
            cnt = 0
            for a, b in pal_pairs:                       # tâm: mọi chữ số đều được (a == b)
                if left == 0 and n > 1 and a == "0":
                    continue
                buf[left], buf[right] = a, b
                cnt += dfs(left + 1, right - 1)
            return cnt

        total += dfs(0, n - 1)
    return total


for low, high in [("0", "100"), ("10", "1000"), ("123", "4567")]:
    brute = sum(1 for x in range(int(low), int(high) + 1) if str(x) == str(x)[::-1])
    got = count_palindromes_in_range(low, high)
    assert got == brute
    print(f"Palindrome trong [{low}, {high}]: {got}")

Palindrome trong [0, 100]: 19
Palindrome trong [10, 1000]: 99
Palindrome trong [123, 4567]: 123


## 16. Bài tập tự luyện

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý theo 3 mức ở mục 17; **đáp án ở cuối notebook** (mục 18) — hãy tự làm trước.

### Bài 1 — Đoán output (không chạy code)

```python
print(rotate("1968"))
print(rotate("619"))
print(is_strobogrammatic("6"))
print("88" < "100", (2, "88") < (3, "100"))
print(count_full_length(6), count_full_length(7))
```

### Bài 2 — Tính tay

a) Đáp án của `low = "10"`, `high = "1000"` là bao nhiêu? Chia theo độ dài 2, 3, 4.

b) Lập bảng như mục 9.4 để tính `F("69200")` (độ dài **lẻ** — chú ý ô tâm).

### Bài 3 — Điền TODO: LeetCode 246 bằng hai con trỏ

Viết `is_strobogrammatic_exercise(num)` trả về `True` nếu chuỗi `num` là số strobogrammatic. Không dùng `rotate`; dùng hai con trỏ `i`, `j`.

In [32]:
def is_strobogrammatic_exercise(num: str) -> bool:
    rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
    i, j = 0, len(num) - 1
    # TODO 1: lặp khi i <= j (nhớ: i == j là ô tâm, vẫn phải kiểm)
    # TODO 2: nếu num[i] không có trong rot, hoặc rot[num[i]] != num[j] -> return False
    # TODO 3: tiến i, lùi j
    # TODO 4: đi hết -> return True
    pass


# Tự làm xong rồi mới mở test:
# assert is_strobogrammatic_exercise("69") == True
# assert is_strobogrammatic_exercise("88") == True
# assert is_strobogrammatic_exercise("962") == False
# assert is_strobogrammatic_exercise("161") == False
# assert is_strobogrammatic_exercise("1") == True

### Bài 4 — Tự viết lại: LeetCode 247 bằng DFS từ ngoài vào trong

Viết `find_strobogrammatic_exercise(n)` trả về **list** mọi số strobogrammatic dài `n` (`1 <= n <= 14`), thứ tự tùy ý. Yêu cầu: dùng mảng `buf` và DFS `(left, right)` như Cách 3, nhưng thay vì đếm thì **thêm** `"".join(buf)` vào list kết quả ở mỗi lá.

In [33]:
def find_strobogrammatic_exercise(n: int) -> list:
    # TODO: buf = [""] * n, result = []
    # TODO: dfs(left, right): lá -> result.append(...); không thì thử 5 cặp với 2 luật (tâm, số 0 đứng đầu)
    # TODO: gọi dfs(0, n - 1), trả về result
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert sorted(find_strobogrammatic_exercise(1)) == ["0", "1", "8"]
# assert sorted(find_strobogrammatic_exercise(2)) == ["11", "69", "88", "96"]
# assert len(find_strobogrammatic_exercise(5)) == 60

### Bài 5 — Biến thể: LeetCode 1056 Confusing Number (Premium)

Đề (tóm tắt theo LeetCode): một **confusing number** là số mà khi xoay 180° thì **mọi chữ số đều xoay được** và kết quả là một số **khác** số ban đầu. Sau khi xoay được phép bỏ các số 0 đứng đầu (xoay `8000` ra `0008`, coi là `8`). Cho số nguyên `n` (`0 <= n <= 10^9`), trả về `True` nếu `n` là confusing number.

Ví dụ: `n = 6` ⇒ `True` (xoay thành `9`); `n = 89` ⇒ `True` (thành `68`); `n = 11` ⇒ `False` (vẫn là `11`). Signature LeetCode: `def confusingNumber(self, n: int) -> bool`.

In [34]:
class ConfusingSolution:
    def confusingNumber(self, n: int) -> bool:
        # TODO 1: đổi n thành chuỗi s; nếu có chữ số không xoay được -> False
        # TODO 2: tạo chuỗi xoay (đọc ngược + lật)
        # TODO 3: so int(chuỗi xoay) với n (int() tự bỏ số 0 đứng đầu)
        pass


# Tự làm xong rồi mới mở test:
# assert ConfusingSolution().confusingNumber(6) == True
# assert ConfusingSolution().confusingNumber(89) == True
# assert ConfusingSolution().confusingNumber(11) == False
# assert ConfusingSolution().confusingNumber(25) == False
# assert ConfusingSolution().confusingNumber(10) == True     # xoay ra "01" = 1

### Bài 6 — Mini challenge: LeetCode 1088 Confusing Number II (Hard, Premium)

Đề (tóm tắt): cho `n` (`1 <= n <= 10^9`), trả về **số lượng** confusing number trong `[1, n]`. Ví dụ: `n = 20` ⇒ `6` (`6, 9, 10, 16, 18, 19`); `n = 100` ⇒ `19`. Signature: `def confusingNumberII(self, n: int) -> int`.

Yêu cầu: **không** duyệt từng số. Gợi ý lớn: một số trong `[1, n]` chỉ gồm chữ `0, 1, 6, 8, 9` thì xoay được; trong số đó, cái nào xoay ra **chính nó** là strobogrammatic. Vậy

```text
confusing(n) = (số trong [1, n] chỉ gồm chữ 0,1,6,8,9) - (số strobogrammatic trong [1, n])
```

Vế đầu đếm được bằng kỹ thuật mục 9.2; vế sau dùng `count_le` của Cách 5 (nhớ `count_le` có tính cả số `0`).

In [35]:
def count_rotatable_le(n: int) -> int:
    # TODO: đếm số trong [1, n] chỉ gồm chữ số 0,1,6,8,9 (không có số 0 đứng đầu)
    #       = (các độ dài ngắn hơn: 4 * 5^(L-1)) + (cùng độ dài: như count_fixed_length_le, chữ đầu cấm 0)
    raise NotImplementedError


def confusing_number_ii(n: int) -> int:
    # TODO: count_rotatable_le(n) - (count_le(str(n)) - 1)   <- trừ 1 vì count_le tính cả số 0
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert confusing_number_ii(20) == 6
# assert confusing_number_ii(100) == 19
# assert confusing_number_ii(1) == 0

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: nhớ "xoay = đọc ngược + lật". `6` đứng một mình xoay thành gì?
- Mức 2: so chuỗi `"88"` với `"100"` chỉ xét ký tự đầu `'8'` vs `'1'`. Tuple so phần tử đầu (độ dài) trước.
- Mức 3: `f(6) = 4 × 5^2`, `f(7) = 4 × 5^2 × 3`.

**Bài 2**
- Mức 1: (a) độ dài 2 và 3 nằm trọn trong khoảng hay không? Số 4 chữ số nhỏ nhất là bao nhiêu?
- Mức 2: (b) `n = 5`, `half = 3`, tập cho phép `[1689, 01689, 018]`; `rest` sau `i = 0` là `5 × 3 = 15`, sau `i = 1` là `3`.
- Mức 3: (b) chữ `2` ở vị trí `i = 2` (ô tâm) không thuộc `{0, 1, 8}` ⇒ dừng ở đó.

**Bài 3**
- Mức 1: giống hệt `is_strobogrammatic` mục 3.3.
- Mức 2: `while i <= j:` (có dấu `=`), bên trong một `if ... return False`.
- Mức 3: `if num[i] not in rot or rot[num[i]] != num[j]: return False` rồi `i += 1; j -= 1`.

**Bài 4**
- Mức 1: chép khung `count_length_dfs`, đổi phần lá: thay `return 1` bằng `result.append("".join(buf))`.
- Mức 2: hàm `dfs` không cần trả gì; `result` là list của hàm ngoài nên hàm trong `append` được mà **không** cần `nonlocal` (chỉ sửa nội dung list, không gán lại biến).
- Mức 3: hai luật: `if left == right and a != b: continue`, `if left == 0 and n > 1 and a == "0": continue`.

**Bài 5**
- Mức 1: dùng lại ý `rotate` mục 3.2.
- Mức 2: `rotated = "".join(rot[c] for c in reversed(s))` sau khi đã kiểm mọi `c in rot`.
- Mức 3: `return int(rotated) != n`.

**Bài 6**
- Mức 1: với `n` có `L` chữ số, số "xoay được" có độ dài `k < L` là `4 × 5^(k-1)` (chữ đầu 4 cách, các chữ sau 5 cách).
- Mức 2: cùng độ dài `L`: duyệt `i` từ trái, tập cho phép ở `i = 0` là `1689`, ở `i > 0` là `01689`; cộng `(số chữ < s[i]) × 5^(L-1-i)`; nếu `s[i]` không thuộc tập thì dừng; đi hết thì `+1`.
- Mức 3: `confusing = rotatable - (count_le(str(n)) - 1)`. Kiểm tra với `n = 100`: `25 - 6 = 19`.

## 18. Đáp án tham khảo

Chỉ xem sau khi đã tự làm!

### Đáp án bài 1

```text
8961          <- "1968" đọc ngược "8691", lật -> "8961"
619           <- 619 là strobogrammatic
False         <- 6 xoay thành 9
False True    <- '8' > '1' nên "88" > "100"; tuple so độ dài 2 < 3 trước
100 300       <- f(6) = 4·5² = 100, f(7) = 4·5²·3 = 300
```

### Đáp án bài 2

a) Độ dài 2: cả 4 số `11, 69, 88, 96` đều `>= 10`. Độ dài 3: nằm **hẳn giữa** ⇒ cả `f(3) = 12`. Độ dài 4: số strobogrammatic 4 chữ số nhỏ nhất là `1001 > 1000` ⇒ 0. Tổng **16**.

b) Độ dài ngắn hơn: `3 + 4 + 12 + 20 = 39`. Bảng cùng độ dài (`n = 5`, `half = 3`):

| i | Tập cho phép | bound[i] | Chữ nhỏ hơn | rest | Cộng | total | Hợp lệ? |
|:-:|---|:-:|---|:-:|:-:|:-:|---|
| 0 | `1 6 8 9` | 6 | `1` | 15 | 15 | 15 | có |
| 1 | `0 1 6 8 9` | 9 | `0, 1, 6, 8` | 3 | 12 | 27 | có |
| 2 (tâm) | `0 1 8` | 2 | `0, 1` | 1 | 2 | 29 | **không** → dừng |

⇒ `F("69200") = 39 + 29 = 68`.

### Đáp án bài 3 → 6

Ô code ngay dưới gồm hai phần: (1) vài dòng `print`/`assert` kiểm chứng đáp án bài 1 và bài 2 ở trên; (2) từ dòng `# Bài 3` trở đi là đáp án bài 3–6, kèm assert ví dụ đề và đối chiếu ngẫu nhiên với brute force (`random.seed(1088)`).

In [36]:
print(rotate("1968"), rotate("619"), is_strobogrammatic("6"))
print("88" < "100", (2, "88") < (3, "100"), count_full_length(6), count_full_length(7))
print("Bài 2a:", strobo_range_math("10", "1000"), "| brute:", strobo_range_bruteforce("10", "1000"))
assert trace_count_le("69200") == F_strobo_brute("69200") == 68


# Bài 3
def answer_is_strobogrammatic_exercise(num: str) -> bool:
    rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
    i, j = 0, len(num) - 1
    while i <= j:
        if num[i] not in rot or rot[num[i]] != num[j]:
            return False
        i += 1
        j -= 1
    return True


# Bài 4
def answer_find_strobogrammatic_exercise(n: int) -> list:
    buf, result = [""] * n, []

    def dfs(left, right):
        if left > right:
            result.append("".join(buf))
            return
        for a, b in PAIRS:
            if left == right and a != b:
                continue
            if left == 0 and n > 1 and a == "0":
                continue
            buf[left], buf[right] = a, b
            dfs(left + 1, right - 1)

    dfs(0, n - 1)
    return result


# Bài 5
class AnswerConfusingSolution:
    def confusingNumber(self, n: int) -> bool:
        rot = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
        s = str(n)
        if any(c not in rot for c in s):
            return False
        rotated = "".join(rot[c] for c in reversed(s))
        return int(rotated) != n


# Bài 6
def answer_count_rotatable_le(n: int) -> int:
    s = str(n)
    L = len(s)
    total = sum(4 * 5 ** (k - 1) for k in range(1, L))       # độ dài ngắn hơn
    for i in range(L):                                       # cùng độ dài
        allowed = "1689" if i == 0 else "01689"
        total += sum(1 for d in allowed if d < s[i]) * 5 ** (L - 1 - i)
        if s[i] not in allowed:
            return total
    return total + 1                                         # chính n xoay được


def answer_confusing_number_ii(n: int) -> int:
    return answer_count_rotatable_le(n) - (count_le(str(n)) - 1)


assert answer_is_strobogrammatic_exercise("69") and answer_is_strobogrammatic_exercise("88")
assert not answer_is_strobogrammatic_exercise("962") and not answer_is_strobogrammatic_exercise("161")
assert answer_is_strobogrammatic_exercise("1")
assert sorted(answer_find_strobogrammatic_exercise(1)) == ["0", "1", "8"]
assert sorted(answer_find_strobogrammatic_exercise(2)) == ["11", "69", "88", "96"]
assert len(answer_find_strobogrammatic_exercise(5)) == 60
cs = AnswerConfusingSolution()
assert cs.confusingNumber(6) and cs.confusingNumber(89) and not cs.confusingNumber(11)
assert not cs.confusingNumber(25) and cs.confusingNumber(10)
assert answer_confusing_number_ii(20) == 6 and answer_confusing_number_ii(100) == 19
assert answer_confusing_number_ii(1) == 0

random.seed(1088)
for n in range(1, 12):
    assert sorted(answer_find_strobogrammatic_exercise(n)) == sorted(find_strobogrammatic(n))
for _ in range(300):
    x = random.randint(0, 10**6)
    assert answer_is_strobogrammatic_exercise(str(x)) == is_strobogrammatic(str(x))
brute_confusing = 0
for x in range(1, 3001):
    brute_confusing += cs.confusingNumber(x)
    if x % 250 == 0 or x < 120:
        assert answer_confusing_number_ii(x) == brute_confusing, x
print("Đáp án bài 3–6: PASS toàn bộ test!")
print("Bài 6 với n = 10^9 (giới hạn đề):", answer_confusing_number_ii(10**9))

8961 619 False
False True 100 300
Bài 2a: 16 | brute: 16
bound = '69200': n = 5, half = 3, độ dài ngắn hơn đóng góp 39
  i=0: cho phép=1689  bound[i]=6 nhỏ hơn=1    rest=15 cộng 15 total=15 -> đi tiếp
  i=1: cho phép=01689 bound[i]=9 nhỏ hơn=0168 rest=3 cộng 12 total=27 -> đi tiếp
  i=2: cho phép=018   bound[i]=2 nhỏ hơn=01   rest=1 cộng 2  total=29 -> DỪNG
  => F('69200') = 39 + 29 = 68
Đáp án bài 3–6: PASS toàn bộ test!
Bài 6 với n = 10^9 (giới hạn đề): 1950627


Giải thích bài 6 bằng tay với `n = 100`:

- Số "xoay được" trong `[1, 100]`: độ dài 1 có `4` số (`1, 6, 8, 9`), độ dài 2 có `4 × 5 = 20` số, độ dài 3 chỉ có `100` (chữ `1, 0, 0` đều xoay được) ⇒ `4 + 20 + 1 = 25`.
- Số strobogrammatic trong `[1, 100]`: `count_le("100") = 7` (gồm `0, 1, 8, 11, 69, 88, 96`), bỏ số `0` ⇒ `6`.
- Confusing: `25 - 6 = 19` ✓ — khớp ví dụ của đề 1088. Nhờ đếm theo chữ số, `n = 10^9` cũng tính tức thì, trong khi brute force phải duyệt 1 tỉ số.

## 19. Template cuối cùng nên nhớ

```python
PAIRS = [("0","0"), ("1","1"), ("6","9"), ("8","8"), ("9","6")]

class Solution:
    def strobogrammaticInRange(self, low: str, high: str) -> int:
        total = 0
        for n in range(len(low), len(high) + 1):          # 1) chỉ các độ dài có thể
            buf = [""] * n
            def dfs(left, right):
                if left > right:                          # 3) lá: so khi CÙNG độ dài
                    s = "".join(buf)
                    if n == len(low) and s < low:   return 0
                    if n == len(high) and s > high: return 0
                    return 1
                cnt = 0
                for a, b in PAIRS:                        # 2) đặt cặp từ ngoài vào trong
                    if left == right and a != b: continue          # tâm: 0/1/8
                    if left == 0 and n > 1 and a == "0": continue  # không 0 đứng đầu
                    buf[left], buf[right] = a, b
                    cnt += dfs(left + 1, right - 1)
                return cnt
            total += dfs(0, n - 1)
        return total
```

Câu thần chú: **"xoay = đọc ngược + lật 6↔9 — 5 cặp, tâm 0/1/8, đầu cấm 0 — so chuỗi chỉ khi cùng độ dài"**.

Bản nâng cao nhớ kèm: `đáp án = F(high) - F(low) + [low hợp lệ]`, với `F(x) = Σ f(k) (k < len x) + đếm nửa đầu theo vị trí khác nhau đầu tiên + 1 ứng viên khi nửa đầu trùng`, và `f(n) = 4 · 5^((n-2)//2) · (3 nếu n lẻ)`, `f(1) = 3`.

## 20. Checklist trước khi submit

- [ ] Bảng cặp đủ 5 cặp `0-0, 1-1, 6-9, 8-8, 9-6` (không phải `6-6`, `9-9`)?
- [ ] Ô tâm (độ dài lẻ) chỉ `0, 1, 8`?
- [ ] Cấm `0` ở **lớp ngoài cùng** khi `n > 1`, nhưng vẫn cho `0-0` ở lớp trong (`1001`) và vẫn đếm số `"0"`?
- [ ] Chỉ duyệt độ dài từ `len(low)` tới `len(high)`?
- [ ] Chỉ so chuỗi `s < low` / `s > high` khi **cùng độ dài** (không so `"69"` với `"100"`)?
- [ ] Trường hợp `len(low) == len(high)`: kiểm **cả hai** biên?
- [ ] Nếu dùng Cách 5: đã `+ [low strobogrammatic]`, có Loại B (ứng viên khi nửa đầu trùng), có dừng khi `bound[i]` không hợp lệ?
- [ ] Đã thử: hai ví dụ đề, `["0","0"]`, `["6","9"]`, `low == high` hợp lệ/không hợp lệ, biên trùng số strobogrammatic, `["0","999999999999999"]`?
- [ ] Độ phức tạp: Cách 3 `O(S · L)` với `S <= 312 499`, bộ nhớ `O(L)` — đủ nhanh cho 15 chữ số.